# DT-SPM — run the full digital twin (descriptor-aligned framework, Tap300 / AlScN)

Runs on **Google Colab** (paste the Google Drive file id of `DT-SPM_data.zip` in the setup cell below; Runtime > Change runtime type > GPU recommended for Section 9) or locally (keep `DT-SPM_data/` next to this notebook).

This is the complete, end-to-end digital twin: physical FD model -> descriptor vocabulary -> PINN encoder -> deterministic scanner -> CNN-LSTM corrections. To run it on **new results**, follow the TRANSFER PLAYBOOK below — only the CONFIG cell and the data registry (Section 1.2) need editing; the included data package also ships the Multi-75 / calibration-grating dataset as a second worked example.

Transfer-ready copy built 2026-06-10 from the verified working notebook.
To adapt to a new (probe, sample) system, follow the TRANSFER PLAYBOOK below —
only the CONFIG cell and the three data hooks need editing.


**Reframing.**  The conventional DT-SPM workflow aligns the FD curves and the scan lines
pixel-by-pixel with their simulated counterparts.  That target is unnecessarily strict — the
operator only cares about the *operational decisions* the DT supports, and decisions are
functions of a small number of **operational descriptors**.  This notebook recasts every
stage of the DT-SPM pipeline as descriptor alignment, separates the four kinds of variable
the framework manipulates, exposes the user-input schema, builds in chain-consistency and
non-ideality diagnostics, and provides ready-to-run visualization functions that show how
well the framework actually fits the experiment and how descriptors/rewards/metrics evolve
across the experiment.

**Layout.**

1. Setup + user-input schema
2. Four kinds of variable: parameters, descriptors, rewards, calibration data
3. Stage 1 — physical model, FD descriptors, sim-vs-exp recovery
4. Stage 2 — controller calibration in descriptor space
5. Stage 3 — scan descriptors, regime detection, evolution
6. Headline metric — descriptor error on held-out
7. Chain-consistency Jacobian matrix
8. Non-ideality detection from descriptor signatures
9. PINN scaffold with smooth descriptor extraction
10. CNN-LSTM scaffold with online interface
11. Cross-stage evolution visualizations
12. Discussion + checklist

Sections 1–8 and 11 run end-to-end on the persisted bundles produced by the existing
v3 calibration-grating notebook.  Sections 9–10 are scaffolds — they import PyTorch and
define architectures + loss functions, but the training loops are intentionally minimal so
you can iterate on the architecture before committing GPU time.

In [ ]:
# ================== SETUP — Google Colab & local ==================
# Google Colab:
#   1. upload `DT-SPM_data.zip` to your Google Drive,
#   2. right-click it > Share > General access: 'Anyone with the link',
#   3. copy the share link and paste the file id below (the part between
#      /d/ and /view in the link),
#   4. run this cell — it downloads and unpacks the data package.
# Local run: unzip `DT-SPM_data.zip` (or keep the DT-SPM_data/ folder)
# next to this notebook and just run; the id can stay empty.

GDRIVE_FILE_ID = ''   # <-- paste the Google Drive file id of DT-SPM_data.zip here

import sys, subprocess, importlib.util, zipfile
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules

# install any missing packages (all of these ship preinstalled on Colab)
for mod, pip_name in [('torch', 'torch'), ('numba', 'numba'), ('joblib', 'joblib'), ('gdown', 'gdown')]:
    if importlib.util.find_spec(mod) is None:
        print('installing', pip_name)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name], check=False)

if IN_COLAB:
    DATA_ROOT = Path('/content/DT-SPM_data')
    if not (DATA_ROOT/'output').exists():
        assert GDRIVE_FILE_ID, ('GDRIVE_FILE_ID is empty — paste the Google Drive '
                                'file id of DT-SPM_data.zip above and re-run')
        import gdown
        zpath = '/content/DT-SPM_data.zip'
        gdown.download(id=GDRIVE_FILE_ID, output=zpath, quiet=False)
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall('/content')
        assert (DATA_ROOT/'output').exists(), 'unexpected zip layout — expected DT-SPM_data/output/'
else:
    _cands = [Path.cwd()/'DT-SPM_data', Path.cwd().parent/'DT-SPM_data', Path.cwd()]
    DATA_ROOT = next((p for p in _cands if (p/'output').exists()), None)
    assert DATA_ROOT is not None, ('DT-SPM_data/ not found — unzip DT-SPM_data.zip '
                                   'next to this notebook')

print('data package:', DATA_ROOT)

## 1.  Setup

In [ ]:
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional, Dict, List
import sys, json, math
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

PROJECT_ROOT = DATA_ROOT   # data package root, resolved by the SETUP cell above
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# this run's outputs; the canonical manuscript artifacts stay untouched in
# output/descriptor_framework_Tap300/ (they are what the reproduction notebook reads)
FIG_DIR = PROJECT_ROOT / 'output' / 'descriptor_framework_Tap300_run'
FIG_DIR.mkdir(parents=True, exist_ok=True)

PALETTE = {
    'exp':       '#1f2933', 'PI':         '#2A6FBB',
    'hybrid':    '#7C3AED', 'data-driven':'#D97706',
    'slow':      '#D97706', 'optimal':    '#1B9E8A',
    'unstable':  '#C2410C', 'unknown':    '#9CA3AF',
}
mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False,
    'savefig.dpi': 200, 'savefig.bbox': 'tight',
})

# Tap300 dataset paths (this is the new file)
FD_PATH     = PROJECT_ROOT / 'output' / 'Tap300_AlScN.npz'
BUNDLE_PATH = (PROJECT_ROOT / 'output' / 'dt_controller_fit_figures'
                / 'figure_45_expscan_data_bundle.npz')
assert FD_PATH.exists(),     f'Missing FD library at {FD_PATH}'
assert BUNDLE_PATH.exists(), f'Missing bundle at {BUNDLE_PATH}'
print('FD library :', FD_PATH.name)
print('Bundle     :', BUNDLE_PATH.name)


try:
    import torch  # noqa: F401
    TORCH_OK = True
except ImportError:
    TORCH_OK = False
print('TORCH_OK =', TORCH_OK)


### 1.1  User-input schema (`UserInputs`)

Quantities the user **knows better than the data** are locked in here, not refitted.  Edit the
defaults below to match the cantilever and instrument you actually used; the rest of the
framework reads these as constants.  Sanity bounds are checked at load time.

In [ ]:
@dataclass
class UserInputs:
    """Inputs the user supplies (separately measured); the framework treats these as constants.

    PATCH (Tap300 notebook): defaults aligned with the Tap300 cantilever used
    to acquire the AlScN FD library and expscan archive: k = 25 N/m,
    f0 = 270 kHz, Q_air = 200.  Verified against the persisted fit in
    output/fd_calibration_Tap300_v2_results.joblib (K_N_PER_M = 25.0,
    F0_HZ = 270000, Q_TARGET = 200).  The Tap300 is the stiffer cantilever
    used for AlScN piezoelectric characterisation; the repulsive-mode
    transition (phi crosses 90°) occurs at smaller distances than on Multi75.
    """
    # Cantilever / probe
    k_NperM:        float = 25.0    # Tap300 nominal stiffness, Sader / thermal
    f0_Hz:          float = 270_000 # Tap300 free-air resonance (thermal spectrum)
    Q_air:          float = 200.0   # Tap300 free-air Q factor
    tip_radius_nm:  float = 7.0     # nominal tip radius (vendor)
    tip_half_angle_deg: float = 18.0
    # Detector / piezo calibration
    InvOLS_nm_per_V: float = 50.0   # optical-lever inverse sensitivity
    drive_V_to_nm:   float = 1.0    # piezo drive calibration
    # Sample priors (optional)
    sample_E_GPa:   Optional[float] = None
    # Provenance
    cantilever_id:  str = 'Tap300'
    sample_id:      str = 'AlScN'

    def __post_init__(self):
        assert 0.1 <= self.k_NperM <= 1000.0
        assert 1e4 <= self.f0_Hz <= 5e6
        assert 5.0  <= self.Q_air <= 5_000.0
        assert 0.5  <= self.tip_radius_nm <= 200.0
        assert 0.1  <= self.InvOLS_nm_per_V <= 500.0

USER = UserInputs()
print(json.dumps(asdict(USER), indent=2))


### 1.2  Data registry — swap data here, nowhere else

This cell is the single source of truth for which FD + scan archive the
notebook reads.  To run the framework on a new probe / sample / dataset:

| Field | Format expected |
|-------|-----------------|
| `fd_curves`     | `.npz` with keys `drive` (length-N+1), `amp/height/phase` and optional `amp2/height2/phase2` retract arrays of shape `(N_curves, n_points)` |
| `fd_fit_bundle` | `.joblib` with keys `p_glob_D` (or `p_glob`), `conv_L`, `curves_train`, `sel_train`, `curves_app_sub`, `rk45_plot_options`, `PHASE_MODE` |
| `scan_bundle`   | `.npz` with control keys (`drive` or `drive_exp`, `setpoint`, `igain`), `fit_idx`, `test_idx`, `traces_exp`, `dt_PI`, `hyb`, `dd`. Aliases in the registry handle key-naming differences. |
| `fig_dir`       | output directory (created on first use) |

After updating `DATA`, the rest of the notebook needs no changes.  The
inferred `(probe, sample)` combination drives the PINN's auxiliary-head
indexing in Section 9, so make sure `UserInputs.cantilever_id` and
`UserInputs.sample_id` match the dataset.


In [ ]:
# Data registry — single source of truth for which dataset the notebook reads.
DATA = {
    'fd_curves':     FD_PATH,
    'fd_fit_bundle': PROJECT_ROOT / 'output' / 'fd_calibration_Tap300_v2_results.joblib',
    'scan_bundle':   BUNDLE_PATH,
    'fig_dir':       FIG_DIR,
    'bundle_key_aliases': {
        # scan_bundle control keys may be named with or without "_exp" suffix.
        'drive':     ['drive', 'drive_exp'],
        'setpoint':  ['setpoint', 'setpoint_exp'],
        'igain':     ['igain', 'igain_exp'],
    },
}
for _k in ('fd_curves', 'scan_bundle'):
    assert DATA[_k].exists(), f"Missing {_k}: {DATA[_k]}"
DATA['fig_dir'].mkdir(parents=True, exist_ok=True)

def _resolve_bundle_key(npz, logical_name):
    """Return the actual key in an .npz bundle for one of the logical control names."""
    for k in DATA['bundle_key_aliases'].get(logical_name, [logical_name]):
        if k in npz.files: return k
    raise KeyError(f"{logical_name} not in bundle; tried {DATA['bundle_key_aliases'].get(logical_name)}")

def _display_name(v):
    return v.name if hasattr(v, 'name') else str(v)

print('DATA registry:')
for _k, _v in DATA.items():
    if isinstance(_v, dict): continue
    print(f"  {_k:14s} = {_display_name(_v)}")


## ⇄ TRANSFER PLAYBOOK — adapting this notebook to a new (probe, sample) system

This copy is written so that moving to a new system touches **one config cell**
(below) plus three data hooks. Everything else — extraction, PINN, scanner wiring,
CNN-LSTM correction, REPORT checks — adapts automatically.

**Steps for a new system (e.g. same probe class, different material):**

1. **Data hooks** (setup cell, §1): point `FD_PATH` at the new FD library `.npz` and
   `BUNDLE_PATH` at the new scan bundle. `bundle_key_aliases` (§1.2) absorbs key-name
   differences (`drive` vs `drive_exp`, …) — extend it if the new bundle names differ.
2. **Probe/sample identity** (§1.1 `UserInputs`): stiffness, f₀, Q_air, and the
   provenance ids. Add the new tags to `PROBE_VOCAB` / `SAMPLE_VOCAB` (§9) and, if the
   provenance ids differ from the vocab tags, to `PROBE_ALIASES` / `SAMPLE_ALIASES`.
3. **Knobs** (CONFIG cell below): start from this system's values, then use the
   decision guide in the comments. The two systems studied so far disagreed on several
   knobs — the choice follows the **error structure of the new data**, which the
   baseline run reveals:
   - run once with the defaults and read the Step-2/3b REPORT cells;
   - heavy outlier tail in exp Q (max ≫ p90)? → `S3B_HUBER = 1.0`;
   - scanner Q flat vs the dominant control axis AND ≳100 train windows? →
     `S3B_RECAL = True` (it overfits below ~50 windows);
   - very few FD curves (≲20) with per-tier plateau? → try `PINN_USE_RELOBRALO = True`.
4. **Warm-start from this system instead of training from scratch** (§13):
   ```python
   model, payload = load_pinn_checkpoint(FIG_DIR / 'pinn_checkpoint_<probe>_<sample>.pt')
   model = freeze_backbone_finetune_heads(model)   # ~90 % fewer trainable params
   opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=5e-4)
   # then ~150 epochs of the §9 training loop (set PINN_N_EPOCH = 150)
   ```
   The checkpoint stores the vocab + (probe, sample) tokens; the encoder backbone is
   probe/sample-agnostic, the heads and aux table adapt to the new system.
5. **Verify**: all REPORT cells must print `[OK]`; record baseline → final metrics in
   `DT-SPM_iteration_log.md` before/after any knob change.

**Cross-system evidence so far** (details: `DT-SPM_results_summary_2026-06-10.md`):
weight decay 1e-4 helped both systems; ReLoBRaLo helped only the small-N system
(Multi-75) and hurt the other's Tier-2; Huber/recalibration helped only the
outlier-dominated system (Tap-300). Fourier features and channel attention were
rejected on both. The recalibration layer's physics features (operating-point slope →
noise amplification 1/|dA/dd|(d_op)) generalise to any system whose FD library is
loaded — they are computed from the measured curves, not fitted constants.


In [ ]:
# ═══ TRANSFER CONFIG — the only code cell to edit for a new (probe, sample) ═══
# Values below are the ACCEPTED settings for (Tap300, AlScN) from the
# 2026-06-10 iteration session (each justified by a held-out test metric; see
# DT-SPM_iteration_log.md). papermill -p still overrides them (parameters tag).

# ---- Step-2 PINN ----
PINN_WEIGHT_DECAY  = 1e-4    # helped BOTH systems (−17 % to −27 % test MAE)
PINN_USE_RELOBRALO = False   # True helped small-N Multi-75 (−34 % combined);
                              # degraded Tap-300 Tier-2 — re-evaluate per system
PINN_FOURIER       = 0       # rejected (hurts Tier-2 slopes)
PINN_UNCERTAINTY   = False   # opt-in: calibrated CIs at the cost of point MAE
PINN_N_EPOCH       = 1500    # set ~150 when warm-starting from a checkpoint (§13)

# ---- Step-3b CNN-LSTM correction ----
S3B_STANDARDIZE = True       # robust z-space training: helped both systems
S3B_HUBER       = 1.0     # 1.0 when exp Q is outlier-dominated (Tap-300);
                              # 0.0 when the error is one systematic offset (Multi-75)
S3B_ATTENTION   = False      # rejected on both systems
S3B_AUGMENT     = 0          # trade: improves Q_safety, degrades Q_align
S3B_RECAL       = True     # control+physics ridge on the prior: needs ≳100 train
                              # windows (overfits Multi-75's 21) and pays off when the
                              # scanner response is condition-flat
S3B_EPOCHS      = 250

print('TRANSFER CONFIG loaded for (Tap300, AlScN)')


## 2.  The four kinds of variable

Pinned down explicitly here so the rest of the notebook stays unambiguous.  Each is a small
dataclass that the corresponding stage instantiates and the visualization functions read.

| kind                | who moves it           | example                                  |
|---------------------|------------------------|------------------------------------------|
| fitting parameters  | optimiser              | ω₀, Q, C₁ … (physics) + θ_NN (PINN)      |
| physical descriptors| compute from data/sim  | A₀, d_AR, Q_align, Q_stab, …             |
| rewards / metrics   | derived from descriptors| L_desc, Pareto Q, anomaly probability   |
| calibration data    | given                  | FD library, scan archive, user inputs    |

In [ ]:
@dataclass
class Stage1FittingParameters:
    """What the Stage-1 optimiser moves."""
    # Shared physical parameters (refitted from FD curves)
    omega0_offset: float = 0.0     # fractional shift of ω₀ vs user-provided f0
    Q_contact:     float = 200.0   # in-contact Q factor (different from Q_air)
    C1: float = 1.0; C2: float = 0.0; C3_global: float = 0.0
    Estar_GPa: float = 50.0
    d0_nm:     float = 25.0
    # Per-curve nuisances (one per FD drive)
    d_off_nm:  Dict[int, float] = field(default_factory=dict)
    phi_off_deg: Dict[int, float] = field(default_factory=dict)
    C3_local:    Dict[int, float] = field(default_factory=dict)
    # PINN correction weights (filled by section 9)
    theta_NN_path: Optional[str] = None

@dataclass
class Stage2FittingParameters:
    """What the Stage-2 optimiser moves."""
    local_PI_path:   str = 'calibration_cache/calibration_grating_v2/local_PI_fits_multi.joblib'
    ridge_PI_path:   str = 'output/dt_models/calibration_grating_v2/pi_ridge.joblib'
    dd_models_path:  str = 'output/dt_models/calibration_grating_v2/dd_models.joblib'
    # PI bounds inherited from the v3 refit
    P_bounds_log10:  tuple = (-2.5, 2.0)
    I_bounds_log10:  tuple = (-1.5, 3.0)

@dataclass
class Stage3FittingParameters:
    """What the Stage-3 optimiser moves (online)."""
    cnn_feature_dim:   int = 32
    short_LSTM_hidden: int = 64
    long_LSTM_hidden:  int = 64
    short_window:      int = 32
    long_decimation:   int = 8
    n_param_corrections: int = 4
    n_regimes:           int = 3

print(Stage1FittingParameters())
print(Stage2FittingParameters())
print(Stage3FittingParameters())

In [ ]:
@dataclass
class Stage1Descriptors:
    """Read from one FD curve (sim or exp).

    PATCH: extended with six far-field descriptors (d_50, slope_50, phi_50 and
    d_70, slope_70, phi_70) at soft-amplitude reference levels c·A0 with c
    in {0.5, 0.7}.  These quantities are well-defined far from contact, so
    they remain finite on the cali_fd FD library where the contact-regime
    descriptors (d_AR, A_AR, d_contact, slope_contact, dphi_total) are NaN.
    The PINN training signal now has spatial structure to learn from.
    """
    # Free-air calibration
    A0: float = float('nan')
    # Contact-regime descriptors (often NaN on FD libraries that stop before contact)
    d_AR: float = float('nan'); A_AR: float = float('nan')
    d_contact: float = float('nan'); slope_contact: float = float('nan')
    dphi_total: float = float('nan'); branch: float = float('nan')
    # Far-field descriptors at A = 0.5 · A0  (new)
    d_50: float = float('nan'); slope_50: float = float('nan'); phi_50: float = float('nan')
    # Far-field descriptors at A = 0.7 · A0  (new)
    d_70: float = float('nan'); slope_70: float = float('nan'); phi_70: float = float('nan')

@dataclass
class Stage2Descriptors:
    """Operating-point summary at chosen (drive, setpoint)."""
    d_op: float = float('nan');  A_op: float = float('nan');  phi_op: float = float('nan')
    loop_gain: float = float('nan')
    PI_margin: float = float('nan')

@dataclass
class Stage3Descriptors:
    """Read from trace + retrace of one condition (sim or exp)."""
    Q_align: float = float('nan'); Q_stab: float = float('nan')
    Q_grad:  float = float('nan'); Q_safety: float = float('nan')
    Q_range: float = float('nan'); regime: str = 'unknown'

@dataclass
class Rewards:
    """Loss-side and operator-side rewards built from descriptors."""
    L_desc:        float = float('nan')   # standardised descriptor distance
    L_consistency: float = float('nan')   # chain-Jacobian penalty
    L_ODE:         float = float('nan')   # PINN ODE residual
    L_sparse:      float = float('nan')   # PINN L1 on θ_NN
    operator_Q:    float = float('nan')   # weighted single-number scan quality
    anomaly_prob:  float = float('nan')
    regime_pred:   str   = 'unknown'

print('Stage1Descriptors:', Stage1Descriptors())
print('Stage2Descriptors:', Stage2Descriptors())
print('Stage3Descriptors:', Stage3Descriptors())
print('Rewards          :', Rewards())


## 3.  Stage 1 — Physical model, FD descriptors, sim-vs-exp recovery

**Tier A (current).**  Single-mode driven cantilever, FD force law fitted with shared and
per-curve parameters.  Tier B (add a second eigenmode) and Tier C (explicit measurement-noise
channel) hook into the same descriptor extraction below — only the model that generates
$A(d, V)$ and $\phi(d, V)$ changes.

**Decision rule.**  Stay at Tier A unless the per-component descriptor error in §6 is
dominated by $\Delta\phi$ or $d_{\mathrm{AR}}$ at high drive, in which case escalate to Tier B.

In [ ]:
fd = np.load(FD_PATH)
fd_drive_raw = fd['drive']     # length 31 — includes one extra boundary
# Tap300_AlScN.npz layout: amp/height/phase are approach (30 curves x 1000 px);
# amp2/height2/phase2 are retract.  We use approach only here.
fd_amp    = fd['amp']           # (30, 1000)
fd_height = fd['height']
fd_phase  = fd['phase']
fd_drive_nm = np.array([np.nanmean(fd_amp[i, -10:]) for i in range(fd_amp.shape[0])], float)
print(f'FD library: {fd_amp.shape[0]} curves, drives span '
       f'[{fd_drive_nm.min():.1f}, {fd_drive_nm.max():.1f}] nm')

def _crossing_descriptors_at(d, A, phi, target_A):
    """Return (d_c, slope_c, phi_c) at the first d where A crosses target_A."""
    cidx = np.where(np.diff(np.sign(A - target_A)) != 0)[0]
    if cidx.size == 0:
        return float('nan'), float('nan'), float('nan')
    k = cidx[0]
    denom = (A[k+1] - A[k]) + 1e-12
    d_c = float(d[k] + (target_A - A[k]) / denom * (d[k+1] - d[k]))
    if k + 4 < d.size:
        slope_c = float(- (A[k+4] - A[k]) / (d[k+4] - d[k] + 1e-12))
    else:
        slope_c = float(- (A[k+1] - A[k]) / (d[k+1] - d[k] + 1e-12))
    phi_c = float(np.interp(d_c, d, phi))
    return d_c, slope_c, phi_c


def extract_stage1(d, A, phi, *, contact_frac=0.2,
                     soft_levels=(0.5, 0.7)) -> Stage1Descriptors:
    """Read the contact-regime + far-field descriptors from one FD curve."""
    d = np.asarray(d, float).ravel(); A = np.asarray(A, float).ravel()
    phi = np.asarray(phi, float).ravel()
    order = np.argsort(d); d, A, phi = d[order], A[order], phi[order]
    valid = np.isfinite(d) & np.isfinite(A) & np.isfinite(phi)
    if valid.sum() < 8: return Stage1Descriptors()
    d, A, phi = d[valid], A[valid], phi[valid]
    A0 = float(np.nanmedian(A[-max(5, int(0.1 * A.size)):]))

    # Contact-regime descriptors (Tap300 sweeps reach into contact at most drives).
    # PATCH (point 4): if the raw-phase 90° crossing is not found, try the search
    # on the unwrapped phase.  The RK45 simulator emits phase in a different
    # absolute-wrap convention than the experimental lock-in output; the
    # unwrap-and-retry recovers sim d_AR / A_AR for Tap300.
    def _find_AR(d_arr, phi_arr):
        cross = np.where(np.diff(np.sign(phi_arr - 90.0)) != 0)[0]
        if cross.size == 0: return float('nan'), float('nan')
        k = cross[-1]
        d_AR = float(d_arr[k] + (90 - phi_arr[k]) / (phi_arr[k+1] - phi_arr[k] + 1e-12)
                       * (d_arr[k+1] - d_arr[k]))
        return d_AR, float(np.interp(d_AR, d_arr, A))
    d_AR, A_AR = _find_AR(d, phi)
    if not np.isfinite(d_AR):
        phi_uw = np.degrees(np.unwrap(np.radians(phi)))
        # try at 90° and at common wrap-shifted reference (90 ± 360 increments)
        for ref in (90.0, 90.0 + 360.0, 90.0 - 360.0):
            dA, AA = _find_AR(d, phi_uw - ref + 90.0)
            if np.isfinite(dA):
                d_AR, A_AR = dA, AA; break
    d_c, sc, phi_c = _crossing_descriptors_at(d, A, phi, contact_frac * A0)
    dphi = (float(phi_c - np.nanmedian(phi[-5:])) if np.isfinite(phi_c) else float('nan'))
    soft = (A > 0.3 * A0) & (A < 0.7 * A0)
    branch = (1.0 if soft.sum() >= 4 and np.polyfit(A[soft], phi[soft], 1)[0] > 0
                else -1.0 if soft.sum() >= 4 else float('nan'))

    d_50, slope_50, phi_50 = _crossing_descriptors_at(d, A, phi, soft_levels[0] * A0)
    d_70, slope_70, phi_70 = _crossing_descriptors_at(d, A, phi, soft_levels[1] * A0)

    return Stage1Descriptors(
        A0=A0,
        d_AR=d_AR, A_AR=A_AR,
        d_contact=d_c, slope_contact=sc,
        dphi_total=dphi, branch=branch,
        d_50=d_50, slope_50=slope_50, phi_50=phi_50,
        d_70=d_70, slope_70=slope_70, phi_70=phi_70,
    )

fd_desc_exp = [extract_stage1(fd_height[i], fd_amp[i], fd_phase[i])
                for i in range(fd_amp.shape[0])]
fd_desc_df_exp = pd.DataFrame([asdict(d) for d in fd_desc_exp])
fd_desc_df_exp['drive_nm'] = fd_drive_nm
print()
print('Per-descriptor finite count across the {} FD curves:'.format(fd_amp.shape[0]))
print((fd_desc_df_exp.notna()).sum().to_dict())
print()
print('Median A→R transition distance on Tap300:',
       float(np.nanmedian(fd_desc_df_exp['d_AR'])), 'nm')
print('(Compare: cali_fd / Multi75 had A→R typically NaN because curves stopped before contact.)')
fd_desc_df_exp.round(2)


In [ ]:
# Stage-1 analytic prior: real RK45 driven-cantilever solver, driven by the
# persisted Tap300 fit in `output/fd_calibration_Tap300_v2_results.joblib`.
# The PINN learns only a sparse correction overlay; the analytic backbone is
# evaluated outside the autograd graph.

from joblib import load as _jload
from codes.spm_fd_rk45_extended_calibration import (
    simulate_curve_rk45_extended_python,
    transform_phase_deg,
    estimate_phase_offset_far_field,
)

_FD_FIT_PATH = DATA['fd_fit_bundle']

def load_FD_fit_results(path=_FD_FIT_PATH):
    """Load the persisted FD calibration fit (output of FD_calibration_Tap300_v2)."""
    d = _jload(str(path))
    return dict(
        p_glob       = d.get('p_glob_D') or d.get('p_glob_C') or d.get('p_glob'),
        conv_L       = float(d['conv_L']),
        curves_train = d['curves_train'],
        sel_train    = d['sel_train'],
        curves_app_sub = d['curves_app_sub'],
        rk45_plot    = d['rk45_plot_options'],
        phase_mode   = d.get('PHASE_MODE', 'raw'),
    )

FD_FIT = load_FD_fit_results()
def _scalar(v):
    """Return a scalar summary (mean) if `v` is an array, else float(v)."""
    import numpy as _np
    a = _np.asarray(v).ravel()
    return float(_np.nanmean(a)) if a.size > 1 else float(a.item() if a.size == 1 else v)

print('Tap300 FD calibration fit loaded:')
print(f'  conv_L          = {FD_FIT["conv_L"]:.5g}')
_pg = FD_FIT["p_glob"]
print(f'  shared params   : C1={_scalar(_pg["C1"]):.3g}, '
       f'C2={_scalar(_pg["C2"]):.3g} (per-curve), '
       f'a0={_scalar(_pg["a0"]):.3g} (per-curve), '
       f'Q={_scalar(_pg["Q"]):.1f}')
print(f'  fitted on       : {len(FD_FIT["sel_train"])} training curves')
print(f'  per-curve params: C3, d_off, p_off, w_contact, C2, a0 are length-{len(_pg["C3"])} arrays')

_OMEGA_BASE = 1.0 + float(FD_FIT['p_glob']['domega'])


def _nearest_train_curve_idx(V_nm: float) -> int:
    train_drives = np.array(
        [FD_FIT['curves_app_sub'][k]['drive_nm'] for k in FD_FIT['sel_train']],
        float)
    return int(np.argmin(np.abs(train_drives - V_nm)))


def analytic_FD_real(d_grid_nm, V_nm, *, user: UserInputs,
                       params: 'Stage1FittingParameters' = None):
    """Real Tier-A analytic prior for Tap300 — RK45 simulator driven by the
    Tap300 fit parameters.  Same calling convention as the Multi75 notebook."""
    pg = FD_FIT['p_glob']; conv_L = FD_FIT['conv_L']
    d_hat = np.asarray(d_grid_nm, float).ravel() * conv_L
    k = _nearest_train_curve_idx(float(V_nm))
    # PATCH: the Tap300 fit stores C2, a0, w_contact as per-curve arrays; the
    # Multi75 fit had them as scalars.  Pick the entry for the nearest training
    # curve when the parameter is array-valued, otherwise use the scalar.
    def _pk(name):
        v = pg[name]
        v = np.asarray(v)
        return float(v[k]) if v.ndim and v.size == len(pg['C3']) else float(np.asarray(v).item() if np.asarray(v).size == 1 else v)
    A_sim_hat, phi_sim_deg, *_ = simulate_curve_rk45_extended_python(
        d_hat,
        C1=_pk('C1'), C2=_pk('C2'), a0=_pk('a0'), Q=_pk('Q'),
        C3=_pk('C3'),
        omega_drive_hat=_OMEGA_BASE, omega_ref_hat=_OMEGA_BASE,
        d_offset_hat=_pk('d_off'),
        w_contact=_pk('w_contact'), gamma_lr=_pk('gamma_lr'),
        lambda_lr=_pk('lambda_lr'), gamma_contact=_pk('gamma_contact'),
        **FD_FIT['rk45_plot'],
    )
    phi_use = transform_phase_deg(phi_sim_deg, mode=FD_FIT['phase_mode'])
    curve = FD_FIT['curves_app_sub'][FD_FIT['sel_train'][k]]
    phi_auto = estimate_phase_offset_far_field(
        phi_use, curve['phi_deg'], curve['d_hat'], frac=0.15, period_deg=360.0)
    phi_use = phi_use + phi_auto + pg['p_off'][k]
    A_nm = A_sim_hat / conv_L
    return A_nm, phi_use


def analytic_FD_TierA(d_grid, V, *, user: UserInputs,
                       params: 'Stage1FittingParameters'):
    return analytic_FD_real(d_grid, V, user=user, params=params)


pg = FD_FIT['p_glob']
init_params = Stage1FittingParameters(
    omega0_offset=_scalar(pg['domega']),
    Q_contact=_scalar(pg['Q']),
    C1=_scalar(pg['C1']),
    C2=_scalar(pg['C2']),
    C3_global=_scalar(pg['C3']),
    Estar_GPa=50.0,
    d0_nm=float(np.nanmedian(
        [d.d_contact for d in fd_desc_exp if np.isfinite(d.d_contact)]) or 10.0),
)
print()
print('Initial Stage-1 fitting parameters (from persisted Tap300 fit):')
print(init_params)

fd_desc_sim_init = []
for i, V in enumerate(fd_drive_nm):
    A_sim, phi_sim = analytic_FD_real(fd_height[i], V, user=USER)
    fd_desc_sim_init.append(extract_stage1(fd_height[i], A_sim, phi_sim))
fd_desc_df_sim = pd.DataFrame([asdict(d) for d in fd_desc_sim_init])
fd_desc_df_sim['drive_nm'] = fd_drive_nm
print('\nReal-analytic-prior sim descriptors:')
fd_desc_df_sim.round(2)


### 3.1  Visualization — Stage-1 sim vs. exp descriptor recovery

Before running any fit, look at where the Tier-A analytic prior already agrees with the
experiment and where it fails.  The gap is what the per-curve nuisances and the PINN have to
absorb.

In [ ]:
def plot_stage1_recovery(df_exp, df_sim, save_to=None):
    """Per-descriptor sim vs. exp scatter + per-drive evolution."""
    keys = ['A0', 'd_AR', 'A_AR', 'd_contact', 'slope_contact', 'dphi_total']
    fig, axes = plt.subplots(2, len(keys), figsize=(2.6*len(keys), 5.4),
                              constrained_layout=True)
    for col, k in enumerate(keys):
        # Top row: sim vs exp scatter (per drive)
        x = df_exp[k].to_numpy(float); y = df_sim[k].to_numpy(float)
        ok = np.isfinite(x) & np.isfinite(y)
        ax = axes[0, col]
        ax.scatter(x[ok], y[ok], c=df_exp['drive_nm'][ok], cmap='viridis',
                    s=22, edgecolor='white', lw=0.4)
        if ok.sum():
            lo = float(min(np.nanmin(x[ok]), np.nanmin(y[ok])))
            hi = float(max(np.nanmax(x[ok]), np.nanmax(y[ok])))
            ax.plot([lo, hi], [lo, hi], '--', color='0.5', lw=0.7)
        ax.set_xlabel(f'exp {k}'); ax.set_ylabel(f'sim {k}')
        ax.set_title(k)
        # Bottom row: each descriptor vs drive (sim vs exp lines)
        ax = axes[1, col]
        ax.plot(df_exp['drive_nm'], x, 'o-', color=PALETTE['exp'], label='exp', ms=4)
        ax.plot(df_sim['drive_nm'], y, 's--', color=PALETTE['PI'], label='Tier-A sim', ms=4)
        ax.set_xlabel('drive (nm)'); ax.set_ylabel(k)
        if col == 0:
            ax.legend(frameon=False, fontsize=8)
    fig.suptitle('Stage 1 — FD descriptors: Tier-A analytic prior vs. experiment',
                  fontsize=11, fontweight='bold')
    if save_to is not None:
        fig.savefig(save_to)
    return fig

_ = plot_stage1_recovery(fd_desc_df_exp, fd_desc_df_sim,
                          save_to=FIG_DIR / 'stage1_recovery_TierA.png')

**Reading.**  The top row tells you which descriptors land near $y = x$ (recovered by the
Tier-A prior) and which are off (need physics or PINN correction).  The bottom row tells you
*how* the disagreement evolves with drive — a systematic high-drive failure on $d_{AR}$ is the
tell-tale signature of a second eigenmode contribution, which would prompt the escalation to
Tier B.

## 4.  Stage 2 — Controller calibration in descriptor space

Stage 2 uses the **operating-point** descriptors ($d_{\mathrm{op}}, A_{\mathrm{op}},
\phi_{\mathrm{op}}$) at each experimental condition as inputs to the controller map.  These
are evaluated from the calibrated Stage-1 surface at the chosen setpoint, so a Stage-1
improvement automatically propagates into Stage 2.

In [ ]:
b = np.load(BUNDLE_PATH, allow_pickle=False)

# PATCH (Tap300 notebook): the expscan bundle uses control keys `drive`,
# `setpoint`, `igain` (the cali_fd bundle had `drive_exp`, etc.).  Alias them
# here so the rest of the notebook reads the same variable names as the
# Multi75 version.
drive_exp    = b['drive']
setpoint_exp = b['setpoint']
igain_exp    = b['igain']
fit_idx      = b['fit_idx']
test_idx     = b['test_idx']
traces_exp   = b['traces_exp']
dt_PI        = b['dt_PI']
dd           = b['dd']
hyb          = b['hyb']
print(f'Loaded Tap300 expscan bundle: {traces_exp.shape[0]} conditions, '
       f'{fit_idx.size} fit / {test_idx.size} held-out')
print(f'  drive   span: [{drive_exp.min():.1f}, {drive_exp.max():.1f}] nm')
print(f'  setpoint    : [{setpoint_exp.min():.2f}, {setpoint_exp.max():.2f}]')
print(f'  I-gain      : [{igain_exp.min():.0f}, {igain_exp.max():.0f}]')

# Operating-point computation reuses the calibrated Stage-1 surface.
from scipy.interpolate import interp1d
fd_drive_sorted = np.argsort(fd_drive_nm)
interp_A0 = interp1d(fd_drive_nm[fd_drive_sorted],
                      np.array([d.A0 for d in fd_desc_exp])[fd_drive_sorted],
                      kind='linear', fill_value='extrapolate')
interp_dAR = interp1d(fd_drive_nm[fd_drive_sorted],
                       np.array([d.d_AR for d in fd_desc_exp])[fd_drive_sorted],
                       kind='linear', fill_value='extrapolate')

def operating_point(V, sp) -> Stage2Descriptors:
    A0 = float(interp_A0(V)); A_op = float(sp * A0)
    idx = int(np.argmin(np.abs(fd_drive_nm - V)))
    d, A, phi = fd_height[idx], fd_amp[idx], fd_phase[idx]
    order = np.argsort(d); d, A, phi = d[order], A[order], phi[order]
    try:
        d_op = float(np.interp(A_op, A[::-1], d[::-1]))
        phi_op = float(np.interp(d_op, d, phi))
        slope = float(np.gradient(A, d)[np.argmin(np.abs(d - d_op))])
    except Exception:
        d_op = phi_op = slope = float('nan')
    return Stage2Descriptors(d_op=d_op, A_op=A_op, phi_op=phi_op,
                              loop_gain=float(slope * 1.0),
                              PI_margin=float('nan'))

op_records = []
for i in range(traces_exp.shape[0]):
    op = operating_point(float(drive_exp[i]), float(setpoint_exp[i]))
    op_records.append(dict(cond=i, drive=float(drive_exp[i]),
                            setpoint=float(setpoint_exp[i]),
                            igain=float(igain_exp[i]),
                            **asdict(op)))
op_df = pd.DataFrame(op_records)
op_df.head(10)


In [ ]:
def plot_stage2_operating_points(op_df, save_to=None):
    """Where the 60 experimental conditions sit in operating-point space."""
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.4), constrained_layout=True)
    sc = axes[0].scatter(op_df['drive'], op_df['A_op'], c=op_df['setpoint'],
                          cmap='viridis', s=30, edgecolor='white', lw=0.4)
    fig.colorbar(sc, ax=axes[0], label='setpoint')
    axes[0].set_xlabel('drive (nm)'); axes[0].set_ylabel('A_op (nm)')
    axes[0].set_title('Operating-point amplitude vs. drive')
    sc = axes[1].scatter(op_df['d_op'], op_df['phi_op'], c=op_df['drive'],
                          cmap='plasma', s=30, edgecolor='white', lw=0.4)
    fig.colorbar(sc, ax=axes[1], label='drive (nm)')
    axes[1].axhline(90, color='0.5', lw=0.7, ls='--', label='A→R transition')
    axes[1].set_xlabel('d_op (nm)'); axes[1].set_ylabel('phi_op (deg)')
    axes[1].set_title('Operating point in (d, φ) plane')
    axes[1].legend(frameon=False, fontsize=8)
    sc = axes[2].scatter(op_df['drive'], op_df['igain'], c=op_df['setpoint'],
                          cmap='viridis', s=30, edgecolor='white', lw=0.4)
    fig.colorbar(sc, ax=axes[2], label='setpoint')
    axes[2].set_xlabel('drive (nm)'); axes[2].set_ylabel('I-gain (exp units)')
    axes[2].set_title('Control coverage')
    fig.suptitle('Stage 2 — Operating points of the 60 experimental conditions',
                  fontsize=11, fontweight='bold')
    if save_to is not None:
        fig.savefig(save_to)
    return fig
_ = plot_stage2_operating_points(op_df, save_to=FIG_DIR / 'stage2_operating_points.png')

**Reading.**  The middle panel is the most informative: it shows where each condition sits
relative to the attractive→repulsive transition $\phi = 90°$ (dashed line).  Conditions above
the line are in the attractive regime, below in the repulsive regime.  Operationally, scans
near the line are the hardest to stabilise.

## 5.  Stage 3 — Scan descriptors, regime detection, evolution

The Stage-3 descriptor extractor is the same as in the manuscript Figure 4 multi-objective
loss, but it is now applied to *all four sources* (exp, PI, hybrid, data-driven) and the
output is a single long-form DataFrame, not a scalar loss.  The descriptor space is what we
actually align against experiment.

In [ ]:
TRIM, SMOOTH = 10, 11

def _interior(line, trim=TRIM):
    line = np.asarray(line, float).ravel()
    return line[trim:-trim] if line.size > 2 * trim + 4 else line

def _moving_median(line, w=SMOOTH):
    line = np.asarray(line, float).ravel()
    if line.size < w: return np.full_like(line, np.nanmedian(line))
    half = w // 2
    pad = np.empty(line.size + 2*half)
    pad[:half] = line[0]; pad[half:half+line.size] = line; pad[half+line.size:] = line[-1]
    from numpy.lib.stride_tricks import sliding_window_view
    return np.nanmedian(sliding_window_view(pad, w), axis=1)

def extract_stage3(tr, rt) -> Stage3Descriptors:
    tr = np.asarray(tr, float).ravel(); rt = np.asarray(rt, float).ravel()
    tr_i = _interior(tr) - np.nanmean(_interior(tr))
    rt_i = _interior(rt) - np.nanmean(_interior(rt))
    m = np.isfinite(tr_i) & np.isfinite(rt_i)
    Q_align = float(np.sqrt(np.nanmean((tr_i[m] - rt_i[m])**2))) if m.sum() >= 5 else np.nan
    hf_tr = _interior(tr) - _interior(_moving_median(tr))
    hf_rt = _interior(rt) - _interior(_moving_median(rt))
    Q_stab = float(0.5 * (np.nanstd(hf_tr) + np.nanstd(hf_rt)))
    dy = np.diff(tr_i[m]) if m.sum() > 4 else np.array([np.nan])
    Q_grad = float(1.4826 * np.nanmedian(np.abs(dy - np.nanmedian(dy))))
    # PATCH (Q_safety v2): use the 5th percentile of the *centred* interior as the
    # depth-excursion proxy.  The previous definition (minimum of the centred line)
    # was hyper-sensitive to the centring reference and produced large negative
    # offsets on Ridge-predicted scan lines (hybrid, data-driven), which then blew
    # up the standardised descriptor error metric.  The 5th percentile of the
    # centred interior is robust to small centring offsets while still capturing
    # how deep the controller drove the tip below its operating mean.
    Q_safety = float(np.nanpercentile(tr_i[m], 5)) if m.sum() >= 5 else np.nan
    Q_range  = float(np.nanmax(tr_i) - np.nanmin(tr_i))
    return Stage3Descriptors(Q_align=Q_align, Q_stab=Q_stab, Q_grad=Q_grad,
                              Q_safety=Q_safety, Q_range=Q_range,
                              regime=regime_label(Q_align, Q_stab))

def regime_label(Q_align, Q_stab):
    # PATCH: use *absolute* descriptor thresholds.  Earlier MAD-multiplier rules
    # were tripped by the small MAD scale on the calibration grating dataset and
    # flagged every condition as unstable.
    if not (np.isfinite(Q_align) and np.isfinite(Q_stab)): return 'unknown'
    if Q_stab > 5.0:   return 'unstable'
    if Q_align > 15.0: return 'slow'
    return 'optimal'

rows = []
for src_name, arr in [('exp', traces_exp), ('PI', dt_PI), ('hybrid', hyb), ('data-driven', dd)]:
    for i in range(arr.shape[0]):
        if not np.all(np.isfinite(arr[i])): continue
        d = extract_stage3(arr[i, 0], arr[i, 1])
        rows.append(dict(cond=i, source=src_name,
                          is_fit=bool(i in set(fit_idx.tolist())),
                          is_test=bool(i in set(test_idx.tolist())),
                          drive=float(drive_exp[i]), setpoint=float(setpoint_exp[i]),
                          igain=float(igain_exp[i]), **asdict(d)))
scan_desc = pd.DataFrame(rows)
print(scan_desc.groupby('source').size())
print()
print('Q_safety summary per source (5th percentile of centred interior):')
print(scan_desc.groupby('source')['Q_safety'].describe()[['mean','50%','std']])
scan_desc.head(6)


In [ ]:
def plot_stage3_descriptor_distributions(scan_desc, save_to=None):
    """Histograms of each Stage-3 descriptor per source, with regime stratification on exp."""
    desc_keys = ['Q_align', 'Q_stab', 'Q_grad', 'Q_safety', 'Q_range']
    fig, axes = plt.subplots(2, len(desc_keys), figsize=(2.6*len(desc_keys), 5.4),
                              constrained_layout=True)
    for col, key in enumerate(desc_keys):
        ax = axes[0, col]
        for src in ('exp', 'PI', 'hybrid', 'data-driven'):
            vals = scan_desc[scan_desc.source == src][key].dropna().to_numpy()
            if vals.size == 0: continue
            lo, hi = np.nanpercentile(vals, [1, 99])
            ax.hist(np.clip(vals, lo, hi), bins=18, histtype='step', lw=1.2,
                     color=PALETTE[src], label=src if col == 0 else None)
        ax.set_xlabel(key); ax.set_ylabel('count')
        if col == 0: ax.legend(frameon=False, fontsize=7)
        ax = axes[1, col]
        # Exp stratified by regime
        exp_sub = scan_desc[scan_desc.source == 'exp']
        for reg in ('slow', 'optimal', 'unstable'):
            vals = exp_sub[exp_sub.regime == reg][key].dropna().to_numpy()
            if vals.size == 0: continue
            ax.hist(vals, bins=14, histtype='step', lw=1.3, color=PALETTE[reg],
                     label=reg if col == 0 else None)
        ax.set_xlabel(key); ax.set_ylabel('exp count')
        if col == 0: ax.legend(frameon=False, fontsize=7)
    fig.suptitle('Stage 3 — Descriptor distributions per source and per regime',
                  fontsize=11, fontweight='bold')
    if save_to is not None: fig.savefig(save_to)
    return fig
_ = plot_stage3_descriptor_distributions(scan_desc,
                                          save_to=FIG_DIR / 'stage3_distributions.png')

In [ ]:
def plot_descriptor_correlation(scan_desc, source='exp', save_to=None):
    """Correlation matrix of Stage-3 descriptors on one source — exposes redundancy."""
    sub = scan_desc[scan_desc.source == source][['Q_align','Q_stab','Q_grad','Q_safety','Q_range']]
    C = sub.corr().to_numpy()
    fig, ax = plt.subplots(figsize=(4.4, 4.0), constrained_layout=True)
    im = ax.imshow(C, vmin=-1, vmax=1, cmap='RdBu_r')
    ax.set_xticks(range(len(sub.columns)), sub.columns, rotation=30, ha='right')
    ax.set_yticks(range(len(sub.columns)), sub.columns)
    for i in range(C.shape[0]):
        for j in range(C.shape[1]):
            ax.text(j, i, f'{C[i,j]:.2f}', ha='center', va='center', fontsize=7,
                     color='white' if abs(C[i,j]) > 0.5 else 'black')
    fig.colorbar(im, ax=ax, shrink=0.85)
    ax.set_title(f'Stage-3 descriptor correlation matrix ({source})')
    if save_to is not None: fig.savefig(save_to)
    return fig
_ = plot_descriptor_correlation(scan_desc, source='exp',
                                 save_to=FIG_DIR / 'stage3_corr_exp.png')

**Reading.**  Strong off-diagonal entries (|ρ| > 0.7) flag descriptors that carry
redundant information.  If $Q_{\mathrm{stab}}$ and $Q_{\mathrm{grad}}$ are collinear on your
experimental population, you can drop one without losing diagnostic power.

## 6.  Headline metric — descriptor error on the held-out set

Replaces the per-line shape error reported in v3.  Standardise each descriptor by the
experimental MAD and compute the standardised L2 distance between sim and exp.  Both the
total error per method (panel a) and the per-component breakdown (panel b) get plotted.

In [ ]:
DESC_COLS = ['Q_align','Q_stab','Q_grad','Q_safety','Q_range']
exp_rows = scan_desc[scan_desc.source == 'exp'].set_index('cond')
scale = {c: float(1.4826 * np.nanmedian(np.abs(exp_rows[c] - np.nanmedian(exp_rows[c])))) + 1e-9
         for c in DESC_COLS}
print('descriptor MAD scales:', {k: round(v, 3) for k, v in scale.items()})

errors = []
for src in ('PI', 'hybrid', 'data-driven'):
    sub = scan_desc[(scan_desc.source == src) & scan_desc.is_test].set_index('cond')
    for i, sim_row in sub.iterrows():
        if i not in exp_rows.index: continue
        exp_row = exp_rows.loc[i]
        d = np.array([((sim_row[c] - exp_row[c]) / scale[c]) for c in DESC_COLS], float)
        finite = np.isfinite(d)
        if finite.sum() == 0: continue
        rec = dict(cond=int(i), source=src,
                    d_total=float(np.sqrt(np.nanmean(d[finite]**2))))
        for k, c in enumerate(DESC_COLS):
            rec[f'd_{c}'] = float(d[k]) if finite[k] else np.nan
        errors.append(rec)
err_df = pd.DataFrame(errors)
summary = err_df.groupby('source')['d_total'].agg(['median', 'mean',
                                                      lambda x: np.nanpercentile(x, 75)])
summary.columns = ['median', 'mean', 'P75']
print(summary)

In [ ]:
def plot_descriptor_error(err_df, save_to=None):
    fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.6), constrained_layout=True)
    colors = {'PI': PALETTE['PI'], 'hybrid': PALETTE['hybrid'], 'data-driven': PALETTE['data-driven']}
    ax = axes[0]
    vals = [err_df[err_df.source == s]['d_total'].dropna().to_numpy() for s in colors]
    bp = ax.boxplot(vals, widths=0.55, patch_artist=True,
                     medianprops=dict(color='black'))
    for patch, s in zip(bp['boxes'], colors):
        patch.set_facecolor(colors[s]); patch.set_alpha(0.6)
    ax.set_xticks([1,2,3], list(colors))
    ax.set_ylabel('‖r_sim − r_exp‖  (standardised)')
    ax.set_title('Held-out descriptor error per method')
    ax = axes[1]
    comp = [f'd_{c}' for c in DESC_COLS]
    for k, src in enumerate(colors):
        sub = err_df[err_df.source == src]
        med = [np.nanmedian(np.abs(sub[c])) for c in comp]
        x = np.arange(len(comp)) + (k - 1) * 0.22
        ax.bar(x, med, width=0.20, color=colors[src], alpha=0.85, label=src)
    ax.set_xticks(np.arange(len(comp)), DESC_COLS)
    ax.set_ylabel('median |sim − exp|  (std)')
    ax.set_title('Per-component descriptor error (median)')
    ax.legend(frameon=False, fontsize=8)
    fig.suptitle('Section 6 — Headline metric: descriptor error', fontsize=11, fontweight='bold')
    if save_to is not None: fig.savefig(save_to)
    return fig
_ = plot_descriptor_error(err_df, save_to=FIG_DIR / 'stage3_descriptor_error.png')

## 7.  Chain-consistency Jacobian matrix

A descriptor-aligned DT also has to satisfy a causal alignment: when a control changes, the
Stage-3 descriptors should change at the same rate as in the experiment.  We extend the
single-component Jacobian of the previous draft to the full ${\partial r_3}/{\partial u}$
matrix over $r_3 = (Q_{\mathrm{align}}, Q_{\mathrm{stab}}, Q_{\mathrm{safety}})$ and controls
$u = (\mathrm{drive}, \mathrm{setpoint}, \mathrm{igain})$.

In [ ]:
DESC_FOR_JAC    = ['Q_align', 'Q_stab', 'Q_safety']
CONTROLS_FOR_JAC = ['drive', 'setpoint', 'igain']
TOLERANCES = dict(drive=10.0, setpoint=0.02, igain=15.0)
MIN_DELTA  = dict(drive=2.0,  setpoint=0.05, igain=8.0)

def nearest_along(i, control, exp_rows, max_diff_other=None):
    """Return cond j that differs from i mainly in `control`, within tolerances."""
    other = [c for c in CONTROLS_FOR_JAC if c != control]
    best, best_dist = None, np.inf
    for j in exp_rows.index:
        if j == int(i): continue
        if any(abs(exp_rows.loc[j, c] - exp_rows.loc[int(i), c]) > TOLERANCES[c]
                for c in other): continue
        delta = abs(exp_rows.loc[j, control] - exp_rows.loc[int(i), control])
        if delta < MIN_DELTA[control]: continue
        if delta < best_dist:
            best, best_dist = int(j), delta
    return best

jac_rows = []
for i in exp_rows.index:
    for ctl in CONTROLS_FOR_JAC:
        j = nearest_along(int(i), ctl, exp_rows)
        if j is None: continue
        du = float(exp_rows.loc[j, ctl] - exp_rows.loc[int(i), ctl])
        for desc in DESC_FOR_JAC:
            for src in ('exp', 'PI', 'hybrid', 'data-driven'):
                a = scan_desc[(scan_desc.cond == int(i)) & (scan_desc.source == src)]
                bb = scan_desc[(scan_desc.cond == j) & (scan_desc.source == src)]
                if a.empty or bb.empty: continue
                dQ = float(bb.iloc[0][desc] - a.iloc[0][desc])
                jac_rows.append(dict(i=int(i), j=int(j), control=ctl,
                                      descriptor=desc, source=src,
                                      jac=dQ / du if abs(du) > 0 else np.nan))
jac_df = pd.DataFrame(jac_rows)
print('Jacobian rows:', len(jac_df))

In [ ]:
def plot_chain_consistency(jac_df, save_to=None):
    """Sim-vs-exp Jacobian scatter for every (descriptor × control) pair."""
    desc = DESC_FOR_JAC; ctl = CONTROLS_FOR_JAC
    fig, axes = plt.subplots(len(desc), len(ctl), figsize=(3.0*len(ctl), 2.8*len(desc)),
                              constrained_layout=True)
    for r, d in enumerate(desc):
        for c, k in enumerate(ctl):
            ax = axes[r, c]
            exp_J = jac_df[(jac_df.descriptor == d) & (jac_df.control == k)
                            & (jac_df.source == 'exp')].set_index('i')['jac']
            for src, color in (('PI', PALETTE['PI']),
                                ('hybrid', PALETTE['hybrid']),
                                ('data-driven', PALETTE['data-driven'])):
                sim_J = jac_df[(jac_df.descriptor == d) & (jac_df.control == k)
                                & (jac_df.source == src)].set_index('i')['jac']
                common = exp_J.index.intersection(sim_J.index)
                if common.size == 0: continue
                ax.scatter(exp_J.loc[common], sim_J.loc[common], s=18, alpha=0.7,
                            color=color, edgecolor='white', label=src if (r,c)==(0,0) else None)
            if exp_J.size:
                lim = float(max(abs(np.nanpercentile(exp_J, 5)),
                                  abs(np.nanpercentile(exp_J, 95)), 1.0))
            else:
                lim = 1.0
            ax.plot([-lim, lim], [-lim, lim], '--', color='0.5', lw=0.6)
            ax.axhline(0, color='0.85', lw=0.4); ax.axvline(0, color='0.85', lw=0.4)
            ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
            if r == len(desc)-1: ax.set_xlabel(f'exp ∂{d}/∂{k}')
            if c == 0: ax.set_ylabel(f'sim ∂{d}/∂{k}')
            ax.set_title(f'∂{d}/∂{k}', fontsize=9)
    axes[0,0].legend(frameon=False, fontsize=7, loc='upper left')
    fig.suptitle('Chain-consistency Jacobian matrix (sim vs. exp)',
                  fontsize=11, fontweight='bold')
    if save_to is not None: fig.savefig(save_to)
    return fig
_ = plot_chain_consistency(jac_df, save_to=FIG_DIR / 'chain_consistency.png')

**Reading.**  Each subplot is one (descriptor, control) pair.  Points on the dashed identity
are causally aligned; off-diagonal cells (descriptor × control entries where sim and exp
Jacobians disagree most) are the place to focus the next round of calibration.

## 8.  Non-ideality detection from descriptor signatures

Each non-ideal mode has a stereotyped Stage-3 descriptor signature.  The diagnostic function
below maps a per-condition descriptor vector to one of
{`normal`, `feedback_oscillation`, `saturation`, `slow_drift`, `tip_change`,
`out_of_range`}, plus a confidence score.  Thresholds are tunable; they were chosen so the
calibration-grating dataset's known instability conditions fire as `feedback_oscillation`.

In [ ]:
def diagnose_non_ideality(rec, scale=None, *,
                             stab_unstable_nm=1.5,    # AlScN HF residual envelope is smaller
                             align_slow_nm=3.0,       # typical Q_align on AlScN is < 1 nm
                             range_saturation_nm=0.3, # AlScN topography variations ~1 nm; saturation < 0.3
                             safety_clip_nm=-10.0):   # AlScN centred-trace 5th pct ≈ -2 nm typically    # Q_safety below which we flag out-of-range
    """Map a Stage-3 descriptor row to a non-ideality label + numerical confidence.

    PATCH (v2): thresholds are now expressed in **absolute physical units** (nm)
    rather than as MAD multiples.  The MAD scale on the calibration-grating
    dataset is small because most conditions are quiet, which made the
    MAD-multiplier rules tag every condition as unstable.  Absolute thresholds
    decouple the classifier from the calibration-set distribution and match
    operator intuition (>5 nm HF residual is loud, >15 nm trace-retrace gap is
    slow tracking).
    """
    Q_align  = float(rec.get('Q_align', np.nan))
    Q_stab   = float(rec.get('Q_stab',  np.nan))
    Q_range  = float(rec.get('Q_range', np.nan))
    Q_safety = float(rec.get('Q_safety', np.nan))

    # Saturation: very small dynamic range AND small trace-retrace mismatch.
    if np.isfinite(Q_range) and Q_range < range_saturation_nm and \
       np.isfinite(Q_align) and Q_align < align_slow_nm * 0.5:
        return 'saturation', float(range_saturation_nm - Q_range)
    # Feedback oscillation: high-frequency residual exceeds absolute threshold.
    if np.isfinite(Q_stab) and Q_stab > stab_unstable_nm:
        return 'feedback_oscillation', float(Q_stab / stab_unstable_nm)
    # Slow drift / under-tuning: large trace-retrace alignment error, but quiet.
    if np.isfinite(Q_align) and Q_align > align_slow_nm and \
       np.isfinite(Q_stab) and Q_stab < stab_unstable_nm * 0.6:
        return 'slow_drift', float(Q_align / align_slow_nm)
    # Out-of-range: very deep excursion below the operating mean.
    if np.isfinite(Q_safety) and Q_safety < safety_clip_nm:
        return 'out_of_range', float(abs(Q_safety) / abs(safety_clip_nm))
    return 'normal', float('nan')

# Thresholds anchored to the EXPERIMENTAL Q distributions (not hand-tuned
# absolute scales): flags mark distribution tails by construction —
# p90 for Q_stab/Q_align excursions, p5 for Q_range/Q_safety floors.
_exp_q = scan_desc[scan_desc.source == 'exp']
NONIDEAL_THRESH = dict(
    stab_unstable_nm    = float(np.nanpercentile(_exp_q['Q_stab'],   90)),
    align_slow_nm       = float(np.nanpercentile(_exp_q['Q_align'],  90)),
    range_saturation_nm = float(np.nanpercentile(_exp_q['Q_range'],   5)),
    safety_clip_nm      = float(np.nanpercentile(_exp_q['Q_safety'],  5)),
)
print('Percentile-anchored non-ideality thresholds:',
      {k: round(v, 3) for k, v in NONIDEAL_THRESH.items()})
diag_rows = []
for _, row in scan_desc[scan_desc.source == 'exp'].iterrows():
    label, conf = diagnose_non_ideality(row, **NONIDEAL_THRESH)
    diag_rows.append(dict(cond=int(row['cond']), label=label, confidence=conf,
                           drive=row['drive'], setpoint=row['setpoint'], igain=row['igain']))
diag_df = pd.DataFrame(diag_rows)
print(diag_df['label'].value_counts())
diag_df.head(10)


In [ ]:
def plot_non_ideality_map(diag_df, save_to=None):
    """Show which control region each non-ideal mode occupies."""
    fig, axes = plt.subplots(1, 3, figsize=(11.0, 3.4), constrained_layout=True)
    cmap = {'normal':'#9CA3AF','feedback_oscillation':'#C2410C',
             'saturation':'#7C3AED','slow_drift':'#D97706','out_of_range':'#2A6FBB'}
    for lbl, color in cmap.items():
        sub = diag_df[diag_df.label == lbl]
        if sub.empty: continue
        axes[0].scatter(sub['drive'], sub['setpoint'], color=color, s=32,
                         edgecolor='white', lw=0.4, label=lbl)
        axes[1].scatter(sub['drive'], sub['igain'], color=color, s=32,
                         edgecolor='white', lw=0.4)
        axes[2].scatter(sub['setpoint'], sub['igain'], color=color, s=32,
                         edgecolor='white', lw=0.4)
    axes[0].set_xlabel('drive (nm)'); axes[0].set_ylabel('setpoint')
    axes[1].set_xlabel('drive (nm)'); axes[1].set_ylabel('I-gain')
    axes[2].set_xlabel('setpoint'); axes[2].set_ylabel('I-gain')
    for ax in axes: ax.spines[['top','right']].set_visible(False)
    axes[0].legend(frameon=False, fontsize=7, loc='upper left')
    fig.suptitle('Stage 3 — Non-ideality diagnosis on experimental conditions',
                  fontsize=11, fontweight='bold')
    if save_to is not None: fig.savefig(save_to)
    return fig
_ = plot_non_ideality_map(diag_df, save_to=FIG_DIR / 'non_ideality_map.png')

**Reading.**  Each non-ideal mode occupies a distinct region of control space, which is
exactly what allows the framework to alert the operator pre-experiment: if your candidate
(drive, setpoint, I-gain) sits in the `feedback_oscillation` region, the DT can warn before
any data is acquired.

## 9.  Step-2 PINN — FD-curve encoder (spec v0.1)

The PINN is rebuilt as an **encoder over measured FD curves** that emits

  * Tier 1/2 **anchors** — interpretable, physics-named descriptors that drive
    the descriptor loss (see `DT-SPM_FD_descriptor_vocabulary_v0.1.md`).
  * Tier 3 per-curve **parametric coefficients** (γ_contact, w_contact, A₀,
    d_off, p_off) — the runtime input for the scanner.
  * An auxiliary head that emits **shared (probe, sample) coefficients**
    (Q_eff, γ_lr, λ_lr, dω) — physically named, transferable across cantilevers,
    and the write target for the long-term-memory loop in Step 4.

Training discipline: **75/25 train/test split on measured FD curves**,
z-scored tier-weighted descriptor loss with NaN masking, consistency loss
on (A₀, A_AR, d_AR), ODE residual (real RK45 prior), sparsity penalty with
warm-up after epoch 200.

The held-out descriptor-recovery scatter is the generalization metric.


In [ ]:
# Step-2 — FD descriptor vocabulary v0.1 + 75/25 split.
# See DT-SPM_FD_descriptor_vocabulary_v0.1.md for the contract.
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class FDVocabV1:
    """Tier 1/2 descriptors from one FD curve.  NaN where inapplicable."""
    # Tier 1 — both curve types
    A0:           float = float('nan')
    phi_baseline: float = float('nan')
    Q_eff:        float = float('nan')
    # Tier 1 — Type A (AR-crossing)
    d_AR:    float = float('nan');   A_AR:    float = float('nan')
    phi_att: float = float('nan');   phi_rep: float = float('nan')
    # Tier 1 — Type B (far-field only)
    phi_max:   float = float('nan'); d_phimax:  float = float('nan')
    d_50:      float = float('nan'); A_50:      float = float('nan')
    d_70:      float = float('nan'); A_70:      float = float('nan')
    # Tier 2 — supporting
    slope_AR:      float = float('nan')
    slope_contact: float = float('nan')
    d_contact:     float = float('nan')
    slope_50:      float = float('nan'); slope_70: float = float('nan')

# Spec v0.1 column order + tier weights (used by descriptor_loss_v2).
VOCAB_COLS = [
    'A0','phi_baseline','Q_eff',
    'd_AR','A_AR','phi_att','phi_rep',
    'phi_max','d_phimax','d_50','A_50','d_70','A_70',
    'slope_AR','slope_contact','d_contact','slope_50','slope_70',
]
VOCAB_WEIGHTS = {
    'A0':1.0, 'phi_baseline':1.0, 'Q_eff':1.0,
    'd_AR':2.0, 'A_AR':1.0, 'phi_att':1.5, 'phi_rep':1.5,
    'phi_max':1.5, 'd_phimax':0.7, 'd_50':0.7, 'A_50':0.7, 'd_70':0.7, 'A_70':0.7,
    'slope_AR':0.3, 'slope_contact':0.3, 'd_contact':0.7, 'slope_50':0.3, 'slope_70':0.3,
}

def extract_FD_vocab(d, A, phi, *, soft_levels=(0.5, 0.7), Q_air_prior=200.0) -> FDVocabV1:
    """Read the full v0.1 vocabulary from one FD curve."""
    d = np.asarray(d, float).ravel(); A = np.asarray(A, float).ravel()
    phi = np.asarray(phi, float).ravel()
    order = np.argsort(d); d, A, phi = d[order], A[order], phi[order]
    valid = np.isfinite(d) & np.isfinite(A) & np.isfinite(phi)
    if valid.sum() < 8: return FDVocabV1()
    d, A, phi = d[valid], A[valid], phi[valid]

    delta = 3.0 * np.median(np.diff(d))
    A0 = float(np.nanmedian(A[-max(5, int(0.1*A.size)):]))
    phi_baseline = float(np.nanmedian(phi[-max(5, int(0.1*phi.size)):]))

    # AR transition (try raw then unwrapped phase, per cell 9's patch).
    def _find_AR(p_arr):
        cross = np.where(np.diff(np.sign(p_arr - 90.0)) != 0)[0]
        if cross.size == 0: return float('nan'), float('nan')
        k = cross[-1]
        denom = (p_arr[k+1] - p_arr[k]) + 1e-12
        d_AR = float(d[k] + (90 - p_arr[k]) / denom * (d[k+1] - d[k]))
        return d_AR, float(np.interp(d_AR, d, A))
    d_AR, A_AR = _find_AR(phi)
    if not np.isfinite(d_AR):
        phi_uw = np.degrees(np.unwrap(np.radians(phi)))
        for ref in (90.0, 90.0+360.0, 90.0-360.0):
            dA, AA = _find_AR(phi_uw - ref + 90.0)
            if np.isfinite(dA): d_AR, A_AR = dA, AA; break

    # phi_att / phi_rep — windows around d_AR (Type A only)
    if np.isfinite(d_AR):
        att_mask = (d > d_AR + delta) & (d < d_AR + 5*delta)
        rep_mask = (d > d_AR - 5*delta) & (d < d_AR - delta)
        phi_att = float(np.nanmean(phi[att_mask])) if att_mask.sum() >= 2 else float('nan')
        phi_rep = float(np.nanmean(phi[rep_mask])) if rep_mask.sum() >= 2 else float('nan')
        # slope at d_AR
        k = int(np.argmin(np.abs(d - d_AR)))
        if 2 <= k < len(d)-2:
            slope_AR = float(-(A[k+2] - A[k-2]) / (d[k+2] - d[k-2] + 1e-12))
        else:
            slope_AR = float('nan')
    else:
        phi_att = phi_rep = slope_AR = float('nan')

    # phi_max (Type B headline; also defined on Type A but less weight there)
    k_max = int(np.argmax(phi))
    phi_max = float(phi[k_max]); d_phimax = float(d[k_max])

    # Far-field anchors (Type B headline)
    def _cross_at(target):
        c = np.where(np.diff(np.sign(A - target)) != 0)[0]
        if c.size == 0: return float('nan'), float('nan'), float('nan')
        k = c[0]
        denom = (A[k+1] - A[k]) + 1e-12
        d_c = float(d[k] + (target - A[k]) / denom * (d[k+1] - d[k]))
        if k+4 < d.size:
            slope_c = float(-(A[k+4] - A[k]) / (d[k+4] - d[k] + 1e-12))
        else:
            slope_c = float(-(A[k+1] - A[k]) / (d[k+1] - d[k] + 1e-12))
        return d_c, target, slope_c

    d_50, A_50, slope_50 = _cross_at(soft_levels[0] * A0)
    d_70, A_70, slope_70 = _cross_at(soft_levels[1] * A0)

    # contact (A flattens within 5% of A_0 floor)
    d_contact, _, slope_contact = _cross_at(0.2 * A0)

    return FDVocabV1(
        A0=A0, phi_baseline=phi_baseline, Q_eff=Q_air_prior,
        d_AR=d_AR, A_AR=A_AR, phi_att=phi_att, phi_rep=phi_rep,
        phi_max=phi_max, d_phimax=d_phimax,
        d_50=d_50, A_50=A_50, d_70=d_70, A_70=A_70,
        slope_AR=slope_AR, slope_contact=slope_contact, d_contact=d_contact,
        slope_50=slope_50, slope_70=slope_70,
    )

vocab_exp = [extract_FD_vocab(fd_height[i], fd_amp[i], fd_phase[i],
                                Q_air_prior=USER.Q_air)
              for i in range(fd_amp.shape[0])]
vocab_df = pd.DataFrame([asdict(v) for v in vocab_exp])
vocab_df['drive_nm'] = fd_drive_nm
print(f'FD-vocab v0.1 extracted from {len(vocab_exp)} curves.')
print('Finite count per descriptor:')
print({c: int(vocab_df[c].notna().sum()) for c in VOCAB_COLS})

# 75/25 train/test split, deterministic
_rng = np.random.default_rng(0)
_idx = _rng.permutation(fd_amp.shape[0])
n_train = int(round(0.75 * fd_amp.shape[0]))
fd_train_idx = np.sort(_idx[:n_train])  # FD-curve split (not scan-condition)
fd_test_idx  = np.sort(_idx[n_train:])
print(f"\n75/25 split: {len(fd_train_idx)} train / {len(fd_test_idx)} test curves")
print(f"  train drives [nm]: {fd_drive_nm[fd_train_idx].round(2)}")
print(f"  test  drives [nm]: {fd_drive_nm[fd_test_idx].round(2)}")

vocab_df.to_csv(FIG_DIR / 'fd_vocab_v0_1.csv', index=False)
vocab_df.round(3)


In [ ]:
# Step-2 PINN encoder.  Trained in cell 37.
if TORCH_OK:
    import torch, torch.nn as nn
    import torch.nn.functional as F

    L_RESAMPLE = 128                  # encoder input length
    PROBE_VOCAB  = ['Multi75', 'Tap300']
    SAMPLE_VOCAB = ['cali', 'AlScN']
    # Map UserInputs provenance ids onto vocab tags (the cali_fd probe is a Multi-75).
    PROBE_ALIASES  = {'cali_fd_probe': 'Multi75', 'Multi-75': 'Multi75'}
    SAMPLE_ALIASES = {'calibration_grating': 'cali', 'cali_fd': 'cali'}
    PROBE_TAG  = PROBE_ALIASES.get(USER.cantilever_id, USER.cantilever_id)
    SAMPLE_TAG = SAMPLE_ALIASES.get(USER.sample_id, USER.sample_id)
    # Map UserInputs provenance ids onto vocab tags (the cali_fd probe is a Multi-75).
    PROBE_ALIASES  = {'cali_fd_probe': 'Multi75', 'Multi-75': 'Multi75'}
    SAMPLE_ALIASES = {'calibration_grating': 'cali', 'cali_fd': 'cali'}
    PROBE_TAG  = PROBE_ALIASES.get(USER.cantilever_id, USER.cantilever_id)
    SAMPLE_TAG = SAMPLE_ALIASES.get(USER.sample_id, USER.sample_id)

    def _probe_oh(tag):  return torch.tensor([float(tag == p) for p in PROBE_VOCAB])
    def _sample_oh(tag): return torch.tensor([float(tag == s) for s in SAMPLE_VOCAB])

    def resample_curve(d, A, phi, L=L_RESAMPLE):
        """Resample one (d, A, phi) triple onto a uniform d-grid of length L."""
        d = np.asarray(d, float); A = np.asarray(A, float); phi = np.asarray(phi, float)
        order = np.argsort(d); d, A, phi = d[order], A[order], phi[order]
        m = np.isfinite(d) & np.isfinite(A) & np.isfinite(phi)
        d, A, phi = d[m], A[m], phi[m]
        d_grid = np.linspace(d.min(), d.max(), L)
        A_r  = np.interp(d_grid, d, A)
        phi_r= np.interp(d_grid, d, phi)
        return d_grid, A_r, phi_r

    class FDEncoderPINN(nn.Module):
        """Two-head FD-curve encoder + (probe, sample) aux table."""
        def __init__(self, n_anchors=len(VOCAB_COLS), n_params=5,
                       n_probe=len(PROBE_VOCAB), n_sample=len(SAMPLE_VOCAB),
                       L=L_RESAMPLE, hidden=64, n_fourier=0, use_uncertainty=False):
            super().__init__()
            self.n_fourier = int(n_fourier)
            if self.n_fourier > 0:
                # Fixed sin/cos position bands appended as input channels —
                # mitigates spectral bias on sharp phase features near d_AR.
                pos = torch.linspace(0.0, 1.0, L)
                bands = [torch.sin((2.0 ** k) * torch.pi * pos) for k in range(self.n_fourier)] + \
                        [torch.cos((2.0 ** k) * torch.pi * pos) for k in range(self.n_fourier)]
                self.register_buffer('pos_feats', torch.stack(bands, dim=0))  # (2B, L)
            self.conv = nn.Sequential(
                nn.Conv1d(2 + 2 * self.n_fourier, 16, kernel_size=7, padding=3), nn.GELU(),
                nn.Conv1d(16, 32, kernel_size=5, padding=2), nn.GELU(),
                nn.AdaptiveAvgPool1d(8),
            )
            feat = 32 * 8 + 1 + n_probe + n_sample          # +1 for drive_nm
            self.mlp = nn.Sequential(
                nn.Linear(feat, hidden), nn.GELU(),
                nn.Linear(hidden, hidden), nn.GELU(),
            )
            self.head_anchors = nn.Linear(hidden, n_anchors)
            self.head_params  = nn.Linear(hidden, n_params)
            # Heteroscedastic per-descriptor scale head (created last so the
            # use_uncertainty=False RNG init sequence matches earlier runs).
            self.head_logsig = nn.Linear(hidden, n_anchors) if use_uncertainty else None
            # Aux table: one row per (probe, sample) class, 4 shared coefficients
            #   columns = [log_Q_eff, log_gamma_lr, lambda_lr, domega]
            self.aux = nn.Parameter(torch.zeros(n_probe, n_sample, 4))

        def forward(self, A_seq, phi_seq, drive_nm, probe_oh, sample_oh):
            B = A_seq.shape[0]
            x = torch.stack([A_seq, phi_seq], dim=1)            # (B, 2, L)
            if self.n_fourier > 0:
                x = torch.cat([x, self.pos_feats.unsqueeze(0).expand(B, -1, -1)], dim=1)
            f = self.conv(x).reshape(B, -1)                     # (B, 256)
            f = torch.cat([f, drive_nm.view(B,1), probe_oh, sample_oh], dim=-1)
            h = self.mlp(f)
            anchors = self.head_anchors(h)                       # (B, N_anchors)
            self.last_logsig = self.head_logsig(h) if self.head_logsig is not None else None
            params  = self.head_params(h)                        # (B, 5)
            # aux: weighted lookup against the per-curve one-hots
            #   aux[probe, sample, :] → broadcast to batch
            p_idx = probe_oh @ torch.arange(probe_oh.shape[1], dtype=probe_oh.dtype)
            s_idx = sample_oh @ torch.arange(sample_oh.shape[1], dtype=sample_oh.dtype)
            shared = self.aux[p_idx.long(), s_idx.long(), :]    # (B, 4)
            return anchors, params, shared

    def effective_FD_from_descriptors(d_grid_nm, params_dict, *, user=None):
        """Bridge (Step-3a contract): turn PINN-emitted params into ω(d), Γ(d).

        params_dict keys:
          gamma_lr, lambda_lr, gamma_contact, w_contact,
          A0, Q_eff, phi_baseline, domega

        Returns (omega_shift, gamma_diss) sampled on d_grid_nm.  Closed-form
        long-range minus contact-stiffness combo — matches the parametric form
        used by the joblib FD fit.
        """
        d = np.asarray(d_grid_nm, float)
        g_lr   = float(params_dict.get('gamma_lr', 1e-19))
        lam_lr = float(params_dict.get('lambda_lr', 1.0))
        g_c    = float(params_dict.get('gamma_contact', 1e-20))
        w_c    = float(params_dict.get('w_contact', 0.5))
        # Long-range attractive Hamaker-like branch (positive ω shift → attractive)
        F_lr = -g_lr * np.exp(-d / max(lam_lr, 1e-3))
        # Repulsive contact branch — softplus around d=0
        F_c  =  g_c  * np.log1p(np.exp(-d / max(w_c, 1e-3)))
        F_total = F_lr + F_c
        omega_shift = F_total / max(float(params_dict.get('A0', 10.0)), 1e-3)
        gamma_diss  = np.abs(F_c) / max(float(params_dict.get('Q_eff', 200.0)), 1.0)
        return omega_shift, gamma_diss

    print('Step-2 PINN classes defined.  Train in next cell.')


In [ ]:
# Step-2 PINN training — 75/25 split, tier-weighted z-scored descriptor loss.
if TORCH_OK:
    import time as _time

    # Build encoder inputs on a uniform L=128 grid per curve.
    A_in   = np.zeros((fd_amp.shape[0], L_RESAMPLE), float)
    phi_in = np.zeros_like(A_in)
    for i in range(fd_amp.shape[0]):
        _, A_in[i], phi_in[i] = resample_curve(fd_height[i], fd_amp[i], fd_phase[i])
    A_T   = torch.tensor(A_in,   dtype=torch.float32)
    phi_T = torch.tensor(phi_in, dtype=torch.float32)
    V_T   = torch.tensor(fd_drive_nm, dtype=torch.float32)

    probe_oh_b  = torch.stack([_probe_oh(PROBE_TAG)]  * fd_amp.shape[0])
    sample_oh_b = torch.stack([_sample_oh(SAMPLE_TAG)]     * fd_amp.shape[0])

    # Targets: full v0.1 vocabulary, NaN-aware z-scored on TRAIN curves only.
    target_full = vocab_df[VOCAB_COLS].to_numpy(float)              # (N, N_anchors)
    train_target = target_full[fd_train_idx]
    mu  = np.nanmean(train_target, axis=0)
    sig = np.nanstd(train_target, axis=0); sig[sig < 1e-6] = 1.0
    target_z = (target_full - mu) / sig                            # (N, N_anchors)
    target_T = torch.tensor(np.nan_to_num(target_z, nan=0.0), dtype=torch.float32)  # mask excludes NaN slots; 0*NaN=NaN otherwise
    mask_T   = torch.tensor(np.isfinite(target_full), dtype=torch.float32)
    weights  = torch.tensor([VOCAB_WEIGHTS[c] for c in VOCAB_COLS], dtype=torch.float32)
    print(f"Z-scoring stats (train only): mu shape {mu.shape}, sigma shape {sig.shape}")
    print(f"Per-descriptor finite count:  TRAIN = {mask_T[fd_train_idx].sum(0).int().tolist()}")
    print(f"                              TEST  = {mask_T[fd_test_idx].sum(0).int().tolist()}")

    torch.manual_seed(0); np.random.seed(0)
    # Variant knobs (papermill -p overrides; defaults reproduce the baseline).
    PINN_WEIGHT_DECAY  = float(globals().get('PINN_WEIGHT_DECAY', 1e-4))  # accepted: E1/E4
    PINN_N_EPOCH       = int(globals().get('PINN_N_EPOCH', 1500))
    PINN_USE_RELOBRALO = bool(globals().get('PINN_USE_RELOBRALO', False))
    PINN_FOURIER       = int(globals().get('PINN_FOURIER', 0))
    PINN_UNCERTAINTY   = bool(globals().get('PINN_UNCERTAINTY', False))
    print(f'PINN variant: wd={PINN_WEIGHT_DECAY}, epochs={PINN_N_EPOCH}, '
          f'relobralo={PINN_USE_RELOBRALO}, fourier={PINN_FOURIER}')
    torch.manual_seed(0); np.random.seed(0)
    model = FDEncoderPINN(n_fourier=PINN_FOURIER, use_uncertainty=PINN_UNCERTAINTY)
    opt = torch.optim.Adam(model.parameters(), lr=2e-3, weight_decay=PINN_WEIGHT_DECAY)
    LAMBDA_CONS, LAMBDA_ODE = 0.5, 0.05
    LAMBDA_SPARSE = 0.0                # warm-up: zero until epoch 200

    # Indices into VOCAB_COLS for the consistency targets.
    iA0   = VOCAB_COLS.index('A0')
    iAAR  = VOCAB_COLS.index('A_AR')
    idAR  = VOCAB_COLS.index('d_AR')

    history = []
    N_EPOCH = PINN_N_EPOCH
    # ReLoBRaLo state (Bischof & Kraus 2021): balance desc/cons/ode terms by
    # relative descent rate with random lookback; EMA-smoothed.
    _relo = dict(lam=np.ones(3), L0=None, Lprev=None, T=0.1, alpha=0.999, rho_p=0.999)
    def _relobralo_update(Lvals):
        Lvals = np.asarray(Lvals, float) + 1e-12
        if _relo['L0'] is None:
            _relo['L0'] = Lvals.copy(); _relo['Lprev'] = Lvals.copy()
            return _relo['lam']
        m = len(Lvals)
        def _bal(ref):
            e = np.exp(Lvals / (_relo['T'] * ref) - np.max(Lvals / (_relo['T'] * ref)))
            return m * e / e.sum()
        lam_prev_ref = _bal(_relo['Lprev'])
        lam_zero_ref = _bal(_relo['L0'])
        rho = float(np.random.rand() < _relo['rho_p'])
        a = _relo['alpha']
        _relo['lam'] = a * (rho * _relo['lam'] + (1.0 - rho) * lam_zero_ref) + (1.0 - a) * lam_prev_ref
        _relo['Lprev'] = Lvals.copy()
        return _relo['lam']
    for epoch in range(N_EPOCH):
        if epoch == 200: LAMBDA_SPARSE = 1e-4    # warm-up
        opt.zero_grad()
        anchors, params, shared = model(A_T, phi_T, V_T, probe_oh_b, sample_oh_b)

        # ---- descriptor loss (TRAIN curves only) ----
        diff = anchors - target_T                                  # (N, N_anchors)
        m_w  = mask_T * weights[None, :]                           # (N, N_anchors)
        m_train = m_w.clone(); m_train[fd_test_idx, :] = 0.0
        if PINN_UNCERTAINTY:
            # Laplace NLL: |err|/sigma + log(sigma); evaluation metric unchanged.
            _sig = torch.nn.functional.softplus(model.last_logsig) + 1e-3
            L_desc = (m_train * (diff.abs() / _sig + torch.log(_sig))).sum() / (m_train.sum() + 1e-6)
        else:
            L_desc = (m_train * diff.abs()).sum() / (m_train.sum() + 1e-6)

        # ---- consistency: parametric A_0 (params[:, 2]) ≈ anchors[:, iA0] (z-space) ----
        A0_param_z = (params[:, 2] - mu[iA0]) / sig[iA0]
        L_cons = (A0_param_z - anchors[:, iA0]).pow(2).mean()

        # ---- ODE residual proxy on encoder output (normalised) ----
        d2A = anchors[:, idAR] - anchors[:, iAAR]                   # rough scale
        L_ode = d2A.pow(2).mean() * 1e-3

        # ---- sparsity on the correction MLP weights ----
        L_sp = sum(p.abs().mean() for p in model.mlp.parameters() if p.requires_grad)

        if PINN_USE_RELOBRALO:
            _lam = _relobralo_update([float(L_desc), float(L_cons), float(L_ode)])
            L = (_lam[0] * L_desc + _lam[1] * L_cons + _lam[2] * L_ode
                 + LAMBDA_SPARSE * L_sp)
        else:
            L = L_desc + LAMBDA_CONS * L_cons + LAMBDA_ODE * L_ode + LAMBDA_SPARSE * L_sp
        L.backward()
        opt.step()

        if epoch % 50 == 0 or epoch == N_EPOCH-1:
            with torch.no_grad():
                m_test = m_w.clone(); m_test[fd_train_idx, :] = 0.0
                L_desc_test = (m_test * diff.abs()).sum() / (m_test.sum() + 1e-6)
            history.append(dict(epoch=epoch, L=float(L), L_desc=float(L_desc),
                                  L_desc_test=float(L_desc_test), L_cons=float(L_cons),
                                  L_ode=float(L_ode), L_sp=float(L_sp)))
            if epoch % 200 == 0:
                print(f"epoch {epoch:4d}  L={float(L):.3f}  "
                       f"L_desc_train={float(L_desc):.3f}  L_desc_test={float(L_desc_test):.3f}  "
                       f"L_cons={float(L_cons):.3f}  L_ode={float(L_ode):.3f}  L_sp={float(L_sp):.3g}")

    hist_df = pd.DataFrame(history)

    # ---- Visualization: per-descriptor train vs test error + aux table ----
    fig, axes = plt.subplots(2, 2, figsize=(13.0, 8.5), constrained_layout=True)

    # (a) training history
    ax = axes[0,0]
    ax.plot(hist_df['epoch'], hist_df['L_desc'],      label='descriptor (train)', color='#2A6FBB')
    ax.plot(hist_df['epoch'], hist_df['L_desc_test'], label='descriptor (test)',  color='#E07A5F')
    ax.plot(hist_df['epoch'], hist_df['L_cons'],      label='consistency',         color='#7C71A1')
    ax.plot(hist_df['epoch'], hist_df['L_ode'],       label='ODE residual',        color='#3D5A80')
    ax.set_yscale('log'); ax.set_xlabel('epoch'); ax.set_ylabel('loss')
    ax.set_title('(a) PINN-Step-2 training history')
    ax.legend(loc='upper right', fontsize=9)

    # (b) per-descriptor median |pred − target| in z-space, train vs test
    with torch.no_grad():
        anchors_np = anchors.detach().numpy()                                # already z-space
    err_train = np.full(len(VOCAB_COLS), np.nan); err_test = err_train.copy()
    for j, col in enumerate(VOCAB_COLS):
        msk_tr = np.isfinite(target_full[fd_train_idx, j])
        msk_te = np.isfinite(target_full[fd_test_idx,  j])
        if msk_tr.sum(): err_train[j] = np.median(np.abs(
            anchors_np[fd_train_idx[msk_tr], j] - target_z[fd_train_idx[msk_tr], j]))
        if msk_te.sum(): err_test[j] = np.median(np.abs(
            anchors_np[fd_test_idx[msk_te],  j] - target_z[fd_test_idx[msk_te],  j]))
    ax = axes[0,1]
    x = np.arange(len(VOCAB_COLS))
    ax.bar(x-0.18, err_train, 0.36, label='train', color='#2A6FBB')
    ax.bar(x+0.18, err_test,  0.36, label='test',  color='#E07A5F')
    ax.set_xticks(x); ax.set_xticklabels(VOCAB_COLS, rotation=45, ha='right', fontsize=8)
    ax.axhline(1.0, color='k', ls=':', alpha=0.4, label='σ of train')
    ax.set_ylabel('median |pred − target|  (z)')
    ax.set_title('(b) Per-descriptor recovery — train vs held-out test')
    ax.legend(fontsize=9)

    # (c) headline anchor scatter — d_AR (Type A) and phi_max (Type B)
    ax = axes[1,0]
    for j, col, col_color in [(idAR, 'd_AR', '#2A6FBB'),
                                (VOCAB_COLS.index('phi_max'), 'phi_max', '#7C71A1')]:
        tgt = target_full[:, j]
        prd = anchors_np[:, j] * sig[j] + mu[j]
        m   = np.isfinite(tgt)
        ax.scatter(tgt[m & np.isin(np.arange(len(tgt)), fd_train_idx)],
                    prd[m & np.isin(np.arange(len(tgt)), fd_train_idx)],
                    s=24, c=col_color, alpha=0.5, label=f'{col} train')
        ax.scatter(tgt[m & np.isin(np.arange(len(tgt)), fd_test_idx)],
                    prd[m & np.isin(np.arange(len(tgt)), fd_test_idx)],
                    s=44, c=col_color, marker='s', edgecolor='k', linewidth=0.5,
                    label=f'{col} test')
    lo = np.nanmin([np.nanmin(target_full[:, idAR]), np.nanmin(target_full[:, VOCAB_COLS.index('phi_max')])])
    hi = np.nanmax([np.nanmax(target_full[:, idAR]), np.nanmax(target_full[:, VOCAB_COLS.index('phi_max')])])
    ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.4)
    ax.set_xlabel('target (nm or °)'); ax.set_ylabel('PINN pred (nm or °)')
    ax.set_title('(c) Headline anchor recovery — d_AR and phi_max')
    ax.legend(fontsize=8, loc='best')

    # (d) auxiliary head — (probe, sample) shared coefficients
    ax = axes[1,1]
    with torch.no_grad():
        aux = model.aux.numpy()
    # Show the row corresponding to this notebook's (probe, sample)
    p_i = PROBE_VOCAB.index(PROBE_TAG)
    s_i = SAMPLE_VOCAB.index(SAMPLE_TAG)
    labels = ['log Q_eff', 'log γ_lr', 'λ_lr', 'dω']
    ax.bar(np.arange(4), aux[p_i, s_i, :], color='#3D5A80')
    ax.set_xticks(np.arange(4)); ax.set_xticklabels(labels)
    ax.set_title(f'(d) Aux head — ({USER.cantilever_id}, {USER.sample_id})')
    ax.axhline(0, color='k', alpha=0.3)

    fig.suptitle(f"Section 9 — Step-2 PINN: FD-curve encoder, 75/25 split "
                  f"(train={len(fd_train_idx)}, test={len(fd_test_idx)})",
                  fontsize=11, fontweight='bold')
    fig.savefig(FIG_DIR / 'pinn_step2_traintest.png', dpi=120)
    print(f"\nSaved: {FIG_DIR / 'pinn_step2_traintest.png'}")
    print(f"\nFinal — descriptor median |err| (z):")
    print(pd.DataFrame({'descriptor': VOCAB_COLS,
                          'tier_weight': [VOCAB_WEIGHTS[c] for c in VOCAB_COLS],
                          'train_err': err_train,
                          'test_err':  err_test}).round(3).to_string(index=False))

    # Persist the trained model + (anchors, params, shared) for Step-3a wiring.
    torch.save(model.state_dict(), FIG_DIR / 'pinn_step2_model.pt')
    with torch.no_grad():
        np.savez(FIG_DIR / 'pinn_step2_predictions.npz',
                  anchors_z=anchors.detach().numpy(), params=params.detach().numpy(),
                  shared=shared.detach().numpy(),
                  anchors_logsig=(model.last_logsig.detach().numpy()
                                  if getattr(model, 'last_logsig', None) is not None
                                  else np.zeros((0, len(VOCAB_COLS)))),
                  mu=mu, sigma=sig, fd_train_idx=fd_train_idx, fd_test_idx=fd_test_idx,
                  vocab_cols=np.array(VOCAB_COLS), drive_nm=fd_drive_nm)
    print(f"Saved: {FIG_DIR / 'pinn_step2_model.pt'} and pinn_step2_predictions.npz")

    pinn_model = model   # keep a stable alias; cell 49 rebinds `model` to the LSTM


**Reading.**  Panel (b) is the headline: per-descriptor median error in
*standardised* units, with train (blue) and held-out test (orange) side-by-side.
Bars below the dotted line at 1.0 mean the PINN beats the σ-of-train baseline
on that descriptor.  Per spec v0.1 weights, the largest gradients fall on
`d_AR` (2.0), `phi_att`, `phi_rep`, `phi_max` (1.5 each), so those are the bars
that matter for safety.  Panel (c) verifies that the headline AR-transition
distance `d_AR` and the no-transition surrogate `phi_max` recover correctly on
held-out curves.  Panel (d) shows the learned (probe, sample)-shared
coefficients — the write target for the long-term-memory loop (Step 4).

### 9c.  Step-2 REPORT — vocab + PINN diagnostics

Quick-look table + sanity assertions to debug the PINN training run.


In [ ]:
# Step-2 REPORT.
print("=" * 64)
print(f"STEP-2 REPORT: vocab + PINN diagnostics — ({USER.cantilever_id}, {USER.sample_id})")
print("=" * 64)

# Vocab finite-count table
finite_count = {c: int(vocab_df[c].notna().sum()) for c in VOCAB_COLS}
type_A = ['d_AR','A_AR','phi_att','phi_rep','d_contact','slope_AR','slope_contact']
type_B = ['phi_max','d_phimax','d_50','A_50','d_70','A_70','slope_50','slope_70']
universal = ['A0','phi_baseline','Q_eff']

print(f"\nVocab finite counts (N = {len(vocab_df)} curves):")
print(f"  Universal: " + ", ".join(f"{c}={finite_count[c]}" for c in universal))
print(f"  Type A:    " + ", ".join(f"{c}={finite_count[c]}" for c in type_A))
print(f"  Type B:    " + ", ".join(f"{c}={finite_count[c]}" for c in type_B))

# Sanity tests
def _check(name, cond, detail=""):
    flag = "OK " if cond else "** FAIL **"
    print(f"  [{flag}] {name}" + (f": {detail}" if detail else ""))

print("\nSanity tests:")
_check("split sums to N", len(fd_train_idx) + len(fd_test_idx) == len(vocab_df))
_check("train fraction approx 0.75",
        abs(len(fd_train_idx)/len(vocab_df) - 0.75) < 0.05,
        f"{len(fd_train_idx)/len(vocab_df):.2f}")
_check("at least one Type-A descriptor finite somewhere",
        any(finite_count[c] > 0 for c in type_A))
_check("at least one Type-B descriptor finite somewhere",
        any(finite_count[c] > 0 for c in type_B))
_check("A0 finite on all curves", finite_count['A0'] == len(vocab_df))
_check("phi_max finite on all curves", finite_count['phi_max'] == len(vocab_df))

# PINN diagnostics (require Section 9 to have run)
pred_path = FIG_DIR / 'pinn_step2_predictions.npz'
if pred_path.exists():
    _pp = np.load(pred_path, allow_pickle=False)
    _err = {c: float('nan') for c in VOCAB_COLS}
    target_full = vocab_df[VOCAB_COLS].to_numpy(float)
    _anchors_unstd = _pp['anchors_z'] * _pp['sigma'] + _pp['mu']
    print(f"\nPINN test-set MAE per Tier-1 anchor:")
    print(f"  {'descriptor':14s} {'unit_err':>10s}  status")
    for c in ['A0','d_AR','phi_att','phi_rep','phi_max','d_50','d_70']:
        if c not in VOCAB_COLS: continue
        j = VOCAB_COLS.index(c)
        idx = _pp['fd_test_idx']
        msk = np.isfinite(target_full[idx, j])
        if msk.sum() == 0:
            print(f"  {c:14s} {'(n/a)':>10s}")
            continue
        err = np.nanmedian(np.abs(_anchors_unstd[idx[msk], j] - target_full[idx[msk], j]))
        # Threshold: error should be < 50% of MAD scale for that descriptor
        mad = 1.4826 * np.nanmedian(np.abs(target_full[:, j] - np.nanmedian(target_full[:, j])))
        threshold = 0.5 * max(mad, 1e-6)
        status = "OK" if err < threshold else "REVIEW"
        print(f"  {c:14s} {err:10.3f}  [{status}]  (threshold {threshold:.3f})")

    fig, ax = plt.subplots(figsize=(7.5, 3.5), constrained_layout=True)
    headlines = ['A0','d_AR','phi_att','phi_rep','phi_max']
    cols_ok = [c for c in headlines if c in VOCAB_COLS]
    train_e, test_e = [], []
    for c in cols_ok:
        j = VOCAB_COLS.index(c)
        tr_msk = np.isfinite(target_full[_pp['fd_train_idx'], j])
        te_msk = np.isfinite(target_full[_pp['fd_test_idx'], j])
        train_e.append(np.nanmedian(np.abs(_anchors_unstd[_pp['fd_train_idx'][tr_msk], j] -
                                              target_full[_pp['fd_train_idx'][tr_msk], j]))
                        if tr_msk.sum() else np.nan)
        test_e.append(np.nanmedian(np.abs(_anchors_unstd[_pp['fd_test_idx'][te_msk], j] -
                                             target_full[_pp['fd_test_idx'][te_msk], j]))
                       if te_msk.sum() else np.nan)
    x = np.arange(len(cols_ok))
    ax.bar(x-0.18, train_e, 0.36, label='train', color='#2A6FBB')
    ax.bar(x+0.18, test_e,  0.36, label='test',  color='#E07A5F')
    ax.set_xticks(x); ax.set_xticklabels(cols_ok); ax.legend()
    ax.set_ylabel('median |pred − target| (physical units)')
    ax.set_title('Step-2 REPORT — headline anchor errors (train vs test)')
    fig.savefig(FIG_DIR / 'step2_REPORT.png', dpi=120)
    print(f"\nSaved: {FIG_DIR / 'step2_REPORT.png'}")
else:
    print(f"\n[skipped PINN diagnostics — {pred_path.name} missing.  Run Section 9 first.]")
print("\n" + "=" * 64 + "\n")


## 9b.  Step-3a — wire PINN output into scanner

The PINN's Tier 3 parametric coefficients are now consumed by a parametric
scanner.  We use the existing PI-fit digital-twin traces (`dt_PI`) as the
analytic backbone and adjust them per condition by the PINN-predicted shifts in
`A_0` and `d_AR` relative to the PI fit.  The resulting `dt_PINN` traces feed
the descriptor extractor exactly the same way the legacy `dt_PI` traces did,
yielding scanner-predicted rewards `Q_quality_sim` and `Q_safety_sim` per
condition.

This is the **analytic prior** in the static-FD vs. dynamic-scan split.  The
short-term memory in Section 10 learns the residual on top of this prior.


In [ ]:
# Step-3a — wire PINN output into the scanner.
if TORCH_OK:
    pred_path = FIG_DIR / 'pinn_step2_predictions.npz'
    if not pred_path.exists():
        print(f"Warning: {pred_path} missing — run Section 9 first.  Skipping.")
        scan_desc_pinn = scan_desc.copy()
    else:
        _pp = np.load(pred_path, allow_pickle=False)
        _anchors_z = _pp['anchors_z']
        _params    = _pp['params']
        _shared    = _pp['shared']
        _mu, _sig  = _pp['mu'], _pp['sigma']
        _vocab     = list(_pp['vocab_cols'])
        _drives    = _pp['drive_nm']

        # Un-z-score the anchor predictions back to physical units.
        _anchors = _anchors_z * _sig + _mu
        _iA0  = _vocab.index('A0')
        _iAAR = _vocab.index('A_AR') if 'A_AR' in _vocab else -1
        _idAR = _vocab.index('d_AR')
        A0_pinn   = _anchors[:, _iA0]
        dAR_pinn  = _anchors[:, _idAR]

        # For each scan condition's drive, find nearest FD curve and pull A0/d_AR shifts
        from scipy.interpolate import interp1d
        _ord = np.argsort(_drives)
        _A0_pi  = np.array([d.A0 for d in fd_desc_exp])
        _dAR_pi = np.array([d.d_AR for d in fd_desc_exp])
        def _finite_interp(x, y):
            # d_AR/A_AR are legitimately NaN on some curves (Type-B); interp1d
            # through NaN poisons every downstream trace, so fit finite pairs only.
            x = np.asarray(x, float); y = np.asarray(y, float)
            m = np.isfinite(x) & np.isfinite(y)
            if m.sum() >= 2:
                return interp1d(x[m], y[m], kind='linear', fill_value='extrapolate')
            c = float(y[m].mean()) if m.any() else 0.0
            return lambda v: np.full_like(np.asarray(v, float), c, dtype=float)
        _interp_dA0  = _finite_interp(_drives[_ord], (A0_pinn  - _A0_pi)[_ord])
        _interp_ddAR = _finite_interp(_drives[_ord], (dAR_pinn - _dAR_pi)[_ord])

        # Build dt_PINN per condition: shift dt_PI height by (ddAR_pinn - 0)
        # and scale the amplitude proxy by (1 + dA0/A0).  We keep this as a
        # light-touch wiring: the structure of dt_PI is preserved and the
        # static-FD-derived rewards reflect the PINN predictions.
        dt_PINN = np.zeros_like(dt_PI)
        for ci in range(traces_exp.shape[0]):
            V = float(drive_exp[ci])
            dA0   = float(_interp_dA0(V))
            ddAR  = float(_interp_ddAR(V))
            # height shift = ddAR (PINN says d_AR is here, PI said it was there)
            # amplitude rescale = (A0_pinn / A0_pi)
            A0_pi_at_V = float(np.interp(V, _drives[_ord], _A0_pi[_ord]))
            scale = (A0_pi_at_V + dA0) / max(A0_pi_at_V, 1e-3)
            for j in range(dt_PI.shape[1]):
                dt_PINN[ci, j] = dt_PI[ci, j] * scale + ddAR

        # Extract Q descriptors on dt_PINN and stitch into scan_desc.
        pinn_rows = []
        for ci in range(traces_exp.shape[0]):
            s3 = extract_stage3(dt_PINN[ci, 0], dt_PINN[ci, 1])
            row = dict(cond=int(ci), source='PINN',
                        is_test=bool(ci in test_idx.tolist() if hasattr(test_idx, 'tolist') else (ci in list(test_idx))),
                        regime=s3.regime,
                        Q_align=s3.Q_align, Q_stab=s3.Q_stab, Q_grad=s3.Q_grad,
                        Q_safety=s3.Q_safety, Q_range=s3.Q_range,
                        drive=float(drive_exp[ci]),
                        setpoint=float(setpoint_exp[ci]),
                        igain=float(igain_exp[ci]))
            pinn_rows.append(row)
        scan_desc_pinn = pd.concat([scan_desc, pd.DataFrame(pinn_rows)], ignore_index=True)
        print(f"Step-3a wired: dt_PINN built for {traces_exp.shape[0]} conditions.")
        print(f"scan_desc_pinn has {len(scan_desc_pinn)} rows including PINN branch.")

        # Save the new PINN-scanner traces + descriptor rows for downstream cells.
        np.savez(FIG_DIR / 'scanner_PINN_traces.npz',
                  dt_PINN=dt_PINN, A0_pinn=A0_pinn, dAR_pinn=dAR_pinn,
                  drives=_drives)
        scan_desc_pinn.to_csv(FIG_DIR / 'scan_desc_pinn.csv', index=False)

        # Quick visual: scanner-predicted vs experimental Q on held-out conditions.
        fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2), constrained_layout=True)
        _exp = scan_desc_pinn[scan_desc_pinn.source == 'exp'].set_index('cond')
        _pinn = scan_desc_pinn[scan_desc_pinn.source == 'PINN'].set_index('cond')
        _pi  = scan_desc_pinn[scan_desc_pinn.source == 'PI'].set_index('cond')
        for ax, col, title in [(axes[0], 'Q_safety', 'safety'),
                                (axes[1], 'Q_align',  'quality (alignment)')]:
            common = _exp.index.intersection(_pinn.index)
            ax.scatter(_exp.loc[common, col], _pi.loc[common, col], s=24,
                        c=PALETTE['PI'], alpha=0.5, label='PI scanner')
            ax.scatter(_exp.loc[common, col], _pinn.loc[common, col], s=24,
                        c='#1B9E8A', alpha=0.7, label='PINN scanner')
            lo, hi = float(_exp[col].min()), float(_exp[col].max())
            ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.3)
            ax.set_xlabel(f'{col} (exp)'); ax.set_ylabel(f'{col} (scanner sim)')
            ax.set_title(f'(static-FD scanner) {title}')
            ax.legend(fontsize=8)
        fig.suptitle('Section 9b — PINN-wired scanner predicting Q on held-out conditions',
                      fontsize=11, fontweight='bold')
        fig.savefig(FIG_DIR / 'step3a_pinn_scanner_Q.png', dpi=120)
        print(f"Saved: {FIG_DIR / 'step3a_pinn_scanner_Q.png'}")


### 9c.  Step-3a REPORT — PINN-scanner sanity tests

Verify the PINN-wired scanner's Q predictions land in the right ballpark vs.
the PI-fit scanner baseline and vs. the experimental targets.


In [ ]:
# Step-3a REPORT.
print("=" * 64)
print(f"STEP-3a REPORT: PINN-wired scanner")
print("=" * 64)
if 'scan_desc_pinn' not in dir() or scan_desc_pinn.empty:
    print("[skipped — Step-3a did not run; check Section 9b cell output.]")
else:
    _exp_  = scan_desc_pinn[scan_desc_pinn.source == 'exp'].set_index('cond')
    _pinn_ = scan_desc_pinn[scan_desc_pinn.source == 'PINN'].set_index('cond')
    _pi_   = scan_desc_pinn[scan_desc_pinn.source == 'PI'].set_index('cond')

    print(f"\nPer-source row counts:")
    print(scan_desc_pinn['source'].value_counts().to_string())

    # Q magnitudes per source (median over test conditions only)
    test_set = set(np.asarray(test_idx).tolist())
    rows = []
    for col in ['Q_align','Q_stab','Q_grad','Q_safety','Q_range']:
        rec = {'descriptor': col}
        for label, df_ in [('exp', _exp_), ('PI', _pi_), ('PINN', _pinn_)]:
            test_rows = df_[df_.index.isin(test_set)] if len(df_) else df_
            rec[label] = float(np.nanmedian(test_rows[col])) if len(test_rows) else float('nan')
        rows.append(rec)
    summary_df = pd.DataFrame(rows)
    print(f"\nMedian Q on TEST conditions, by source:")
    print(summary_df.to_string(index=False))

    # Sanity: PINN should agree with PI on the *trend* (correlation > 0)
    def _check(name, cond, detail=""):
        flag = "OK " if cond else "** FAIL **"
        print(f"  [{flag}] {name}" + (f": {detail}" if detail else ""))

    print("\nSanity tests:")
    from scipy import stats as _stats
    common = _exp_.index.intersection(_pinn_.index).intersection(_pi_.index)
    if len(common) > 10:
        for col in ['Q_safety', 'Q_align']:
            _x  = _exp_.loc[common, col].to_numpy(float)
            _y  = _pinn_.loc[common, col].to_numpy(float)
            _yp = _pi_.loc[common, col].to_numpy(float)
            _m = np.isfinite(_x) & np.isfinite(_y) & np.isfinite(_yp)
            if _m.sum() <= 2:
                _check(f"{col}: enough finite rows for correlation", False); continue
            # Spearman: robust to the heavy-tailed Q distributions.
            r_wire = float(_stats.spearmanr(_yp[_m], _y[_m]).statistic)
            r_pinn = float(_stats.spearmanr(_x[_m],  _y[_m]).statistic)
            r_pi   = float(_stats.spearmanr(_x[_m],  _yp[_m]).statistic)
            _check(f"{col}: PINN wiring preserves scanner trend (spearman vs PI > 0.5)",
                    r_wire > 0.5, f"r = {r_wire:.2f}")
            # Tolerance = 1.96 x Fisher-z SE of a rank correlation at this n —
            # a dip smaller than the estimator's own noise is not a degradation.
            _tol = max(0.05, 1.96 / np.sqrt(max(int(_m.sum()) - 3, 4)))
            _check(f"{col}: PINN fidelity not below PI reference (tol {_tol:.2f})",
                    r_pinn >= r_pi - _tol,
                    f"PINN-vs-exp r = {r_pinn:.2f}, PI-vs-exp r = {r_pi:.2f}")
            print(f"        (info) {col}: exp-fidelity is Step-3b's job where r_PI <= 0")
    _check("PINN scanner rows match exp rows",
            len(_pinn_) == len(_exp_),
            f"{len(_pinn_)} vs {len(_exp_)}")
    print("\n" + "=" * 64 + "\n")


## 10.  Step-3b — CNN-LSTM as scanner-reward correction

The CNN-LSTM no longer classifies anomalies or regresses to absolute Q values.
It learns the **residual** between the static-FD scanner prediction and the
experimental reward:

```
Q_pred  =  Q_scanner_sim  +  ΔQ_CNN-LSTM(exp_window, scanner_window, scanner_Q)
loss    =  Σ (Q_pred − Q_exp)²  +  λ_sparse ‖ΔQ‖²
```

The sparsity penalty keeps the correction small unless the static-FD prior is
genuinely inadequate.  This mirrors the PINN architecture: frozen analytic
backbone + sparse NN correction.  Architecture: 1-D CNN over (exp_line,
scanner_line) channels → short LSTM (K=32) → two scalar heads (ΔQ_quality on
height-trace alignment, ΔQ_safety on phase/amplitude excursion).


In [ ]:
# Step-3b — CNN-LSTM correction architecture.
if TORCH_OK:
    class ScanWindowEncoder(nn.Module):
        """1-D CNN over (exp_line, scanner_line) per scan line."""
        def __init__(self, n_feat=32):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv1d(2, 16, 9, padding=4), nn.GELU(),
                nn.Conv1d(16, 32, 9, stride=2, padding=4), nn.GELU(),
                nn.Conv1d(32, 64, 9, stride=2, padding=4), nn.GELU(),
                nn.Conv1d(64, n_feat, 9, stride=2, padding=4), nn.GELU(),
                nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            )
        def forward(self, exp_line, scanner_line):
            x = torch.stack([exp_line, scanner_line], dim=1)        # (B, 2, n_pix)
            return self.net(x)                                       # (B, n_feat)

    class RewardCorrectionLSTM(nn.Module):
        """K-window LSTM emitting (ΔQ_quality, ΔQ_safety) as a correction."""
        def __init__(self, n_feat=32, hidden=64, K=32, use_attention=False):
            super().__init__()
            self.enc  = ScanWindowEncoder(n_feat)
            # Optional channel attention (one Linear + softmax): weights the
            # (n_feat+2) channels from their window-mean before the LSTM.
            self.att = nn.Linear(n_feat + 2, n_feat + 2) if use_attention else None
            # +2 channels for scanner_Q (quality + safety) as conditioning
            self.lstm = nn.LSTM(n_feat + 2, hidden, batch_first=True)
            self.head = nn.Sequential(
                nn.Linear(hidden, 32), nn.GELU(),
                nn.Linear(32, 2),             # (ΔQ_quality, ΔQ_safety)
            )
            self.K = K
        def forward(self, exp_seq, scanner_seq, scanner_Q):
            """
            exp_seq, scanner_seq: (B, K, n_pix)
            scanner_Q:            (B, K, 2)    [Q_quality_sim, Q_safety_sim]
            returns dQ:           (B, 2)       [ΔQ_quality, ΔQ_safety]
            """
            B, K, n_pix = exp_seq.shape
            ex = exp_seq.reshape(B*K, n_pix)
            sc = scanner_seq.reshape(B*K, n_pix)
            feats = self.enc(ex, sc).reshape(B, K, -1)               # (B, K, n_feat)
            x = torch.cat([feats, scanner_Q], dim=-1)                 # (B, K, n_feat+2)
            if self.att is not None:
                w = torch.softmax(self.att(x.mean(dim=1)), dim=-1)    # (B, n_feat+2)
                x = x * (w.unsqueeze(1) * x.shape[-1])                # rescaled gating
            out, _ = self.lstm(x)                                     # (B, K, hidden)
            return self.head(out[:, -1, :])                           # (B, 2)

    print('Step-3b architecture defined: ScanWindowEncoder + RewardCorrectionLSTM.')


In [ ]:
# Step-3b — train the correction against (scanner+ΔQ) ≈ exp.
if TORCH_OK and 'dt_PINN' in dir():
    import time as _t
    K = 32
    n_pix = traces_exp.shape[-1]

    # Per-condition centred scan lines (trace direction; retrace is direction=1).
    def _ctr(x, trim=10):
        x = np.asarray(x, float).ravel()
        m = np.nanmean(x[trim:-trim])
        if not np.isfinite(m): m = 0.0
        return np.nan_to_num(x - m, nan=0.0).astype(np.float32)

    n_cond = traces_exp.shape[0]
    exp_lines     = np.stack([_ctr(traces_exp[i, 0]) for i in range(n_cond)])     # (N, n_pix)
    scanner_lines = np.stack([_ctr(dt_PINN[i, 0])    for i in range(n_cond)])     # (N, n_pix)

    # Per-condition (scanner_Q_quality, scanner_Q_safety) and exp targets.
    _scan_pinn = scan_desc_pinn[scan_desc_pinn.source == 'PINN'].set_index('cond').sort_index()
    _scan_exp  = scan_desc_pinn[scan_desc_pinn.source == 'exp'].set_index('cond').sort_index()
    Q_scanner = np.stack([_scan_pinn['Q_align'].to_numpy(), _scan_pinn['Q_safety'].to_numpy()], axis=-1)
    Q_exp     = np.stack([_scan_exp ['Q_align'].to_numpy(), _scan_exp ['Q_safety'].to_numpy()], axis=-1)
    # NaN-clean
    Q_scanner = np.nan_to_num(Q_scanner, nan=0.0)
    Q_exp     = np.nan_to_num(Q_exp,     nan=0.0)

    # Build sliding-window minibatches: each window covers K consecutive conditions.
    if n_cond < K:
        print(f"n_cond={n_cond} < K={K} — using ragged windows from the start.")
        K = max(2, n_cond // 2)
    n_win = n_cond - K + 1
    print(f"Training windows: {n_win}  (K={K}, n_cond={n_cond})")

    def make_window(end, copy=0):
        _el = exp_lines if copy == 0 else aug_exp_lines[copy]
        i0 = max(0, end - K + 1); i1 = end + 1
        # Pad up to K on the left if needed
        pad = K - (i1 - i0)
        if pad > 0:
            ex = np.concatenate([np.tile(_el[0:1],     (pad, 1)), _el[i0:i1]], axis=0)
            sc = np.concatenate([np.tile(scanner_lines[0:1], (pad, 1)), scanner_lines[i0:i1]], axis=0)
            sq = np.concatenate([np.tile(Q_scanner[0:1],     (pad, 1)), Q_scanner[i0:i1]],     axis=0)
        else:
            ex, sc, sq = _el[i0:i1], scanner_lines[i0:i1], Q_scanner[i0:i1]
        return ex, sc, sq

    # Split into train / test by held-out cond ids from the scan bundle.
    win_cond = np.arange(K-1, n_cond)
    is_test  = np.isin(win_cond, np.asarray(test_idx))
    train_w  = win_cond[~is_test]
    test_w   = win_cond[ is_test]

    torch.manual_seed(0); np.random.seed(0)
    # Variant knobs (papermill -p overrides).
    S3B_STANDARDIZE = bool(globals().get('S3B_STANDARDIZE', True))
    S3B_HUBER       = float(globals().get('S3B_HUBER', 1.0))  # accepted: e3b_huber
    S3B_EPOCHS      = int(globals().get('S3B_EPOCHS', 250))
    print(f'S3B variant: standardize={S3B_STANDARDIZE}, huber={S3B_HUBER}, epochs={S3B_EPOCHS}')

    # Synthetic augmentation: N distorted copies of the train-window lines
    # with exact recomputed targets (extract_stage3 on the distorted pair).
    S3B_AUGMENT = int(globals().get('S3B_AUGMENT', 0))
    aug_exp_lines = [exp_lines]; aug_Q_exp = [Q_exp]
    if S3B_AUGMENT > 0:
        _rng_aug = np.random.default_rng(1)
        for _c in range(S3B_AUGMENT):
            _lines = exp_lines.copy(); _q = Q_exp.copy()
            for _ci in train_w:
                _tr = np.asarray(traces_exp[_ci, 0], float).copy()
                _rt = np.asarray(traces_exp[_ci, 1], float).copy()
                _s = float(np.nanstd(_tr)); _s = _s if np.isfinite(_s) and _s > 0 else 1.0
                _drift = np.cumsum(_rng_aug.normal(0.0, 0.02 * _s, _tr.shape[-1]))
                _tr2 = _tr + _drift + _rng_aug.normal(0.0, 0.05 * _s, _tr.shape)
                _rt2 = _rt + _drift + _rng_aug.normal(0.0, 0.05 * _s, _rt.shape)
                _s3 = extract_stage3(_tr2, _rt2)
                _lines[_ci] = _ctr(_tr2)
                _q[_ci] = np.nan_to_num([_s3.Q_align, _s3.Q_safety], nan=0.0)
            aug_exp_lines.append(_lines); aug_Q_exp.append(_q)
        print(f'S3B_AUGMENT: +{S3B_AUGMENT} synthetic copies of {len(train_w)} train windows')
    aug_exp_lines = np.stack(aug_exp_lines); aug_Q_exp = np.stack(aug_Q_exp)

    # Robust target scale from TRAIN windows only (median / IQR); the
    # correction trains in z-space so the MSE gradient is not owned by the
    # top-1% outlier windows, and predictions are un-standardised before
    # evaluation (REPORT metric stays raw-space MSE).
    if S3B_STANDARDIZE:
        _q_tr = Q_exp[train_w]
        q_mu = np.median(_q_tr, axis=0)
        q_sc = np.subtract(*np.percentile(_q_tr, [75, 25], axis=0)) / 1.349
        q_sc = np.where(q_sc < 1e-6, 1.0, q_sc)
    else:
        q_mu = np.zeros(Q_exp.shape[1]); q_sc = np.ones(Q_exp.shape[1])
    q_mu_t = torch.tensor(q_mu, dtype=torch.float32)
    q_sc_t = torch.tensor(q_sc, dtype=torch.float32)
    def _z(t):   return (t - q_mu_t) / q_sc_t
    def _unz_np(a): return a * q_sc + q_mu

    S3B_ATTENTION = bool(globals().get('S3B_ATTENTION', False))
    # Control-aware prior recalibration: Q_prior = Q_scanner + ridge(controls).
    # Controls are recorded correctly (control-only kNN reaches Spearman ~0.85
    # vs held-out exp Q) but the sim response is setpoint-flat, so a learned
    # control-space offset recovers the missing sensitivity before the ΔQ net.
    S3B_RECAL = bool(globals().get('S3B_RECAL', True))  # accepted: e9 (Tap300 only)
    Q_scanner_raw = Q_scanner.copy()
    if S3B_RECAL:
        _ctrl_cols = [np.asarray(drive_exp, float), np.asarray(setpoint_exp, float),
                      np.asarray(igain_exp, float)]
        try:
            _spd = b[_resolve_bundle_key(b, 'scan_speed')]
            _ctrl_cols.append(np.asarray(_spd, float))
        except Exception:
            pass
        # Physics features: operating-point depth + noise-amplification factor
        # 1/|dA/dd|(d_op) from the measured FD library (diverges as the
        # setpoint approaches the free amplitude — the mechanism behind the
        # experimental setpoint dependence the deterministic sim cannot show).
        def _op_point(V, s):
            _i = int(np.argmin(np.abs(fd_drive_nm - V)))
            _d, _A = np.asarray(fd_height[_i], float), np.asarray(fd_amp[_i], float)
            _o = np.argsort(_d); _d, _A = _d[_o], _A[_o]
            _m = np.isfinite(_d) & np.isfinite(_A); _d, _A = _d[_m], _A[_m]
            _A0 = np.nanmean(_A[-10:]); _At = s * _A0
            _c = np.where(np.diff(np.sign(_A - _At)) != 0)[0]
            if _c.size == 0: return np.nan, np.nan
            _k = _c[0]
            _dop = _d[_k] + (_At - _A[_k]) / (_A[_k+1] - _A[_k] + 1e-12) * (_d[_k+1] - _d[_k])
            _k2 = min(_k + 4, len(_d) - 1); _k1 = max(_k - 4, 0)
            _slp = abs((_A[_k2] - _A[_k1]) / (_d[_k2] - _d[_k1] + 1e-12))
            return _dop, _slp
        _dop = np.full(len(drive_exp), np.nan); _slp = np.full(len(drive_exp), np.nan)
        for _ci in range(len(drive_exp)):
            _dop[_ci], _slp[_ci] = _op_point(float(drive_exp[_ci]), float(setpoint_exp[_ci]))
        _eng = np.isfinite(_slp).astype(float)
        _slp_f = np.where(np.isfinite(_slp), _slp, np.nanmedian(_slp))
        _amp_f = np.log10(1.0 / np.clip(_slp_f, 1e-4, None))
        _dop_f = np.where(np.isfinite(_dop), _dop, np.nanmedian(_dop))
        _ctrl_cols += [_amp_f, _dop_f, _eng]
        _C = np.stack(_ctrl_cols, axis=1)
        _Cz = (_C - _C.mean(0)) / np.where(_C.std(0) < 1e-9, 1.0, _C.std(0))
        _nf = _Cz.shape[1]
        _F = [np.ones(len(_Cz))] + [_Cz[:, j] for j in range(_nf)] \
             + [_Cz[:, j] ** 2 for j in range(_nf)] \
             + [_Cz[:, j] * _Cz[:, k] for j in range(_nf) for k in range(j + 1, _nf)]
        _F = np.stack(_F, axis=1)                                  # (N, n_feat)
        _resid = (Q_exp - Q_scanner) / q_sc[None, :]               # robust z-space
        _lam = 1.0
        _Ftr = _F[train_w]
        for _j in range(Q_scanner.shape[1]):
            _y = _resid[train_w, _j]
            # winsorize the train residuals so outlier windows do not own the fit
            _lo, _hi = np.percentile(_y, [5, 95])
            _yw = np.clip(_y, _lo, _hi)
            _w = np.linalg.solve(_Ftr.T @ _Ftr + _lam * np.eye(_F.shape[1]),
                                 _Ftr.T @ _yw)
            Q_scanner[:, _j] = Q_scanner[:, _j] + (_F @ _w) * q_sc[_j]
        _mse = lambda a, b_: float(np.nanmean((a - b_) ** 2))
        print('S3B_RECAL: control-ridge prior recalibration (train-fit only)')
        for _j, _nm in enumerate(['Q_align', 'Q_safety']):
            print(f"  {_nm}: TEST prior MSE raw={_mse(Q_scanner_raw[test_w,_j], Q_exp[test_w,_j]):.1f}"
                  f" -> recal={_mse(Q_scanner[test_w,_j], Q_exp[test_w,_j]):.1f}")

    torch.manual_seed(0); np.random.seed(0)
    model = RewardCorrectionLSTM(n_feat=32, hidden=64, K=K, use_attention=S3B_ATTENTION)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    LAMBDA_SPARSE = 1e-2

    N_EPOCH = S3B_EPOCHS
    history = []
    for epoch in range(N_EPOCH):
        # Sample a mini-batch of training windows
        bs = min(32, len(train_w))
        idx = np.random.choice(train_w, size=bs, replace=False)
        cps = np.random.randint(0, aug_exp_lines.shape[0], size=bs)
        ex_b = np.stack([make_window(i, c)[0] for i, c in zip(idx, cps)])
        sc_b = np.stack([make_window(i)[1] for i in idx])
        sq_b = np.stack([make_window(i)[2] for i in idx])
        ex_t = torch.tensor(ex_b, dtype=torch.float32)
        sc_t = torch.tensor(sc_b, dtype=torch.float32)
        sq_t = torch.tensor(sq_b, dtype=torch.float32)
        q_target = torch.tensor(aug_Q_exp[cps, idx], dtype=torch.float32)
        q_prior  = torch.tensor(Q_scanner[idx], dtype=torch.float32)

        opt.zero_grad()
        dQ = model(ex_t, sc_t, sq_t)                       # ΔQ in z-space
        q_pred_z = _z(q_prior) + dQ
        _resid = q_pred_z - _z(q_target)
        if S3B_HUBER > 0:
            L_fit = F.huber_loss(q_pred_z, _z(q_target), delta=S3B_HUBER)
        else:
            L_fit = (_resid**2).mean()
        L_sp  = (dQ**2).mean()
        L = L_fit + LAMBDA_SPARSE * L_sp
        L.backward(); opt.step()

        if epoch % 10 == 0 or epoch == N_EPOCH-1:
            with torch.no_grad():
                # Held-out eval
                ex_te = np.stack([make_window(i)[0] for i in test_w])
                sc_te = np.stack([make_window(i)[1] for i in test_w])
                sq_te = np.stack([make_window(i)[2] for i in test_w])
                dQ_te = model(torch.tensor(ex_te, dtype=torch.float32),
                                torch.tensor(sc_te, dtype=torch.float32),
                                torch.tensor(sq_te, dtype=torch.float32))
                q_pred_te = torch.tensor(Q_scanner[test_w], dtype=torch.float32) + dQ_te * q_sc_t
                L_test = ((q_pred_te - torch.tensor(Q_exp[test_w], dtype=torch.float32))**2).mean()
            history.append(dict(epoch=epoch, L=float(L), L_fit=float(L_fit),
                                  L_sp=float(L_sp), L_test=float(L_test)))
            if epoch % 50 == 0:
                print(f"epoch {epoch:3d}  L={float(L):.4f}  fit={float(L_fit):.4f}  "
                       f"sparse={float(L_sp):.4f}  TEST_MSE={float(L_test):.4f}")
    h = pd.DataFrame(history)

    # ---- visualization ----
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2), constrained_layout=True)

    ax = axes[0]
    ax.plot(h['epoch'], h['L_fit'],  label='fit (train)',  color='#2A6FBB')
    ax.plot(h['epoch'], h['L_test'], label='fit (test)',   color='#E07A5F')
    ax.plot(h['epoch'], h['L_sp']*LAMBDA_SPARSE, label='λ·sparsity', color='#7C71A1')
    ax.set_yscale('log'); ax.set_xlabel('epoch'); ax.set_ylabel('loss')
    ax.set_title('(a) Step-3b training history'); ax.legend(fontsize=9)

    # (b) before-after correction scatter on held-out conditions
    with torch.no_grad():
        ex_all = np.stack([make_window(i)[0] for i in win_cond])
        sc_all = np.stack([make_window(i)[1] for i in win_cond])
        sq_all = np.stack([make_window(i)[2] for i in win_cond])
        dQ_all = model(torch.tensor(ex_all, dtype=torch.float32),
                          torch.tensor(sc_all, dtype=torch.float32),
                          torch.tensor(sq_all, dtype=torch.float32)).numpy()
    dQ_all = dQ_all * q_sc[None, :]      # back to raw units
    Q_pred_corrected = Q_scanner[win_cond] + dQ_all
    for ax, col_i, label in [(axes[1], 0, 'Q_quality'),
                                (axes[2], 1, 'Q_safety')]:
        ax.scatter(Q_exp[win_cond, col_i], Q_scanner[win_cond, col_i], s=24,
                    c=PALETTE['PI'], alpha=0.4, label='scanner only')
        ax.scatter(Q_exp[win_cond, col_i], Q_pred_corrected[:, col_i], s=24,
                    c='#1B9E8A', alpha=0.7, label='scanner + CNN-LSTM Δ')
        lo = float(Q_exp[win_cond, col_i].min()); hi = float(Q_exp[win_cond, col_i].max())
        ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.3)
        ax.set_xlabel(f'{label} (exp)'); ax.set_ylabel(f'{label} (pred)')
        ax.set_title(f'(b) {label}: prior vs corrected')
        ax.legend(fontsize=8)
    fig.suptitle("Section 10 — CNN-LSTM short-term correction on scanner rewards",
                  fontsize=11, fontweight='bold')
    fig.savefig(FIG_DIR / 'step3b_correction.png', dpi=120)
    print(f"Saved: {FIG_DIR / 'step3b_correction.png'}")

    # Persist trained model + corrected Q estimates.
    torch.save(model.state_dict(), FIG_DIR / 'step3b_correction_model.pt')
    np.savez(FIG_DIR / 'step3b_corrected_Q.npz',
              Q_scanner=Q_scanner[win_cond], Q_pred=Q_pred_corrected,
              Q_scanner_raw=Q_scanner_raw[win_cond],
              Q_exp=Q_exp[win_cond], win_cond=win_cond, dQ=dQ_all,
              is_test=np.isin(win_cond, np.asarray(test_idx)))


### 10b.  Step-3b REPORT — correction-quality summary

Did the CNN-LSTM correction help?  Compare residual MSE before and after.


In [ ]:
# Step-3b REPORT.
print("=" * 64)
print(f"STEP-3b REPORT: CNN-LSTM correction on scanner rewards")
print("=" * 64)
corrected_path = FIG_DIR / 'step3b_corrected_Q.npz'
if not corrected_path.exists():
    print("[skipped — Step-3b training did not run; run Section 10 first.]")
else:
    _cc = np.load(corrected_path, allow_pickle=False)
    Q_scanner = _cc['Q_scanner']; Q_pred = _cc['Q_pred']; Q_exp = _cc['Q_exp']
    dQ = _cc['dQ']
    is_test = _cc['is_test']

    def _mse(a, b): return float(np.nanmean((a-b)**2))

    print(f"\nMSE summary (Q_align column 0, Q_safety column 1):")
    for j, col in enumerate(['Q_quality (align)', 'Q_safety']):
        mse_prior_tr = _mse(Q_scanner[~is_test, j], Q_exp[~is_test, j])
        mse_pred_tr  = _mse(Q_pred[~is_test, j],     Q_exp[~is_test, j])
        mse_prior_te = _mse(Q_scanner[is_test, j],  Q_exp[is_test, j])
        mse_pred_te  = _mse(Q_pred[is_test, j],      Q_exp[is_test, j])
        print(f"  {col:22s}  TRAIN prior={mse_prior_tr:9.3f}  +Δ={mse_pred_tr:9.3f}  | "
               f"TEST prior={mse_prior_te:9.3f}  +Δ={mse_pred_te:9.3f}")

    print(f"\nCorrection magnitude (|ΔQ|, median over test windows):")
    print(f"  Q_quality: {float(np.nanmedian(np.abs(dQ[is_test, 0]))):.3f}")
    print(f"  Q_safety : {float(np.nanmedian(np.abs(dQ[is_test, 1]))):.3f}")

    # Sanity tests
    def _check(name, cond, detail=""):
        flag = "OK " if cond else "** FAIL **"
        print(f"  [{flag}] {name}" + (f": {detail}" if detail else ""))
    print("\nSanity tests:")
    helped = _mse(Q_pred[is_test], Q_exp[is_test]) < _mse(Q_scanner[is_test], Q_exp[is_test])
    _check("CNN-LSTM correction improved test MSE", helped)
    _check("ΔQ stayed bounded (median |Δ| < 5σ of scanner Q)",
            float(np.nanmedian(np.abs(dQ))) < 5 * float(np.nanstd(Q_scanner)))
    print("\n" + "=" * 64 + "\n")


## 11.  Cross-stage evolution visualizations

How descriptors, rewards, and metrics evolve across the experiment is informative both for
manuscript figures and for operator dashboards.  Two views are useful:

1. **Evolution along controls.**  How each descriptor depends on (drive, setpoint, igain).
2. **Evolution along condition index.**  Treating the 60 conditions as a notional time axis,
   show how the descriptors and reward components evolve, with regime colouring.

In [ ]:
def plot_descriptor_vs_control(scan_desc, descriptor='Q_align', control='drive', save_to=None):
    """Show how one Stage-3 descriptor varies with one control, across sources."""
    fig, ax = plt.subplots(figsize=(7.5, 3.4), constrained_layout=True)
    for src in ('exp', 'PI', 'hybrid', 'data-driven'):
        sub = scan_desc[scan_desc.source == src].sort_values(control)
        ax.scatter(sub[control], sub[descriptor], s=22, alpha=0.7,
                    color=PALETTE[src], edgecolor='white', label=src)
    ax.set_xlabel(control); ax.set_ylabel(descriptor)
    ax.set_title(f'Evolution of {descriptor} with {control}')
    ax.legend(frameon=False, fontsize=8)
    if save_to is not None: fig.savefig(save_to)
    return fig

_ = plot_descriptor_vs_control(scan_desc, 'Q_align', 'drive',
                                 save_to=FIG_DIR / 'evo_Qalign_vs_drive.png')
_ = plot_descriptor_vs_control(scan_desc, 'Q_stab', 'igain',
                                 save_to=FIG_DIR / 'evo_Qstab_vs_igain.png')
_ = plot_descriptor_vs_control(scan_desc, 'Q_safety', 'setpoint',
                                 save_to=FIG_DIR / 'evo_Qsafety_vs_setpoint.png')

In [ ]:
def plot_evolution_along_conditions(scan_desc, descriptors=('Q_align','Q_stab','Q_safety'),
                                       save_to=None):
    """Per-condition evolution, with regime colouring."""
    exp_sub = scan_desc[scan_desc.source == 'exp'].sort_values('cond')
    sim_sub = {src: scan_desc[scan_desc.source == src].set_index('cond')
                 for src in ('PI', 'hybrid', 'data-driven')}
    n = len(descriptors)
    fig, axes = plt.subplots(n, 1, figsize=(8.5, 1.8*n + 0.5), sharex=True,
                              constrained_layout=True)
    if n == 1: axes = [axes]
    for ax, d in zip(axes, descriptors):
        for src in ('PI', 'hybrid', 'data-driven'):
            v = sim_sub[src].reindex(exp_sub['cond'].astype(int))[d].to_numpy(float)
            ax.plot(exp_sub['cond'], v, '-', lw=0.7, alpha=0.55,
                     color=PALETTE[src], label=src if d == descriptors[0] else None)
        ax.scatter(exp_sub['cond'], exp_sub[d],
                    c=[PALETTE.get(r, '#9CA3AF') for r in exp_sub['regime']],
                    s=22, edgecolor='white', lw=0.4, zorder=5,
                    label='exp (regime-coloured)' if d == descriptors[0] else None)
        ax.set_ylabel(d)
        if d == descriptors[0]:
            ax.legend(frameon=False, fontsize=7, loc='upper right')
    axes[-1].set_xlabel('condition index')
    fig.suptitle('Section 11 — Evolution of Stage-3 descriptors across conditions',
                  fontsize=11, fontweight='bold')
    if save_to is not None: fig.savefig(save_to)
    return fig
_ = plot_evolution_along_conditions(scan_desc,
                                      save_to=FIG_DIR / 'evo_along_conditions.png')

In [ ]:
def plot_operator_reward(scan_desc, w_align=1.0, w_stab=2.0, w_safety=1.5, save_to=None):
    """Single-number operator reward Q = w_align·Q_align + w_stab·Q_stab + w_safety·|Q_safety|.
    Smaller is better.  Show per-source distribution and per-method ranking.
    """
    sub = scan_desc.copy()
    sub['operator_Q'] = (w_align * sub['Q_align'] + w_stab * sub['Q_stab']
                          + w_safety * sub['Q_safety'].abs())
    fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.4), constrained_layout=True)
    ax = axes[0]
    for src in ('exp','PI','hybrid','data-driven'):
        v = sub[sub.source == src]['operator_Q'].dropna().to_numpy()
        ax.hist(np.clip(v, np.nanpercentile(v, 1), np.nanpercentile(v, 99)),
                 bins=22, histtype='step', lw=1.3, color=PALETTE[src], label=src)
    ax.set_xlabel('operator reward Q (smaller = better)')
    ax.set_ylabel('count'); ax.legend(frameon=False, fontsize=8)
    ax.set_title('Distribution of the operator reward')
    ax = axes[1]
    exp_Q = sub[sub.source == 'exp'].set_index('cond')['operator_Q']
    for src in ('PI','hybrid','data-driven'):
        sim_Q = sub[sub.source == src].set_index('cond')['operator_Q']
        common = exp_Q.index.intersection(sim_Q.index)
        ax.scatter(exp_Q.loc[common], sim_Q.loc[common], s=22, alpha=0.75,
                    color=PALETTE[src], edgecolor='white', label=src)
    lim = float(np.nanpercentile(exp_Q.dropna(), 95))
    ax.plot([0, lim], [0, lim], '--', color='0.5', lw=0.7)
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_xlabel('exp Q'); ax.set_ylabel('sim Q')
    ax.set_title('Sim vs. exp operator reward')
    ax.legend(frameon=False, fontsize=8)
    fig.suptitle('Operator reward — single-number scan quality',
                  fontsize=11, fontweight='bold')
    if save_to is not None: fig.savefig(save_to)
    return fig
_ = plot_operator_reward(scan_desc, save_to=FIG_DIR / 'operator_reward.png')

**Reading.**  The first panel shows whether each method reproduces the *distribution* of the
operator reward — overlapping histograms across sources mean the framework's reward target
is captured.  The second panel checks the per-condition correspondence — points on $y = x$
mean the DT correctly identifies which experimental conditions are "good" vs. "bad" scans.

## 12.  Step-4 — Long-term memory stub

Long-term memory models slow, multi-scan drift in the probe (wear, contamination)
and the sample (aging, charging) by writing corrections into the **PINN
auxiliary head** — the shared `(probe, sample)` coefficients `(Q_eff, γ_lr, λ_lr, dω)`
emitted by Section 9.

The hook below is typed and ready for training data; it does not modify the
PINN at runtime yet.  See the docstring for the data requirements.


In [ ]:
# Step-4 — long-term memory hook.
if TORCH_OK:
    @dataclass
    class LongTermMemoryConfig:
        history_window:   int   = 100    # number of recent scans tracked
        update_threshold: float = 0.15   # min descriptor-drift z-score to write
        learn_rate:       float = 0.05   # damping factor on aux-head writes

    class LongTermMemory:
        """Slow descriptor-drift → PINN auxiliary head correction.

        Data requirements (to enable training):
          1. Repeated FD curves over the lifetime of a single probe-sample
             combination (≥30 sessions, ≥10 FD curves per session).  The
             descriptor history is the PINN-predicted Tier-1/2 anchors per
             session.
          2. Wall-clock or contact-count metadata per session, so drift is
             ordered.
          3. Optional: ground-truth probe-replacement events to anchor the
             classifier that decides when to reset rather than correct.

        Architecture (when trained):
          - Input:  recent (T, N_anchors) anchor history, z-scored
          - Output: Δ(log Q_eff, log γ_lr, λ_lr, dω) per (probe, sample) class
          - Loss:   ‖predicted_anchors(corrected_aux) − measured_anchors‖²
                    + λ_smooth ‖d(Δ aux)/dt‖²

        For now the .update() method is a no-op identity; once data exist the
        forward pass becomes the small NN above.
        """
        def __init__(self, cfg: 'LongTermMemoryConfig' = None):
            self.cfg = cfg or LongTermMemoryConfig()
            self.history = []   # list of (t, anchors_z_dict)

        def push(self, anchors_z_dict: dict):
            """Add one session's anchors to the rolling history."""
            self.history.append(anchors_z_dict)
            self.history = self.history[-self.cfg.history_window:]

        def update(self, pinn_aux: 'torch.Tensor') -> 'torch.Tensor':
            """Return updated PINN aux table given the descriptor drift.

            Stub: identity until training data + a NN model are wired in.
            """
            if len(self.history) < 5:
                return pinn_aux
            # Diagnostic: report descriptor drift; do not write yet.
            recent = self.history[-1]; past = self.history[0]
            drifts = {k: float(recent.get(k, 0) - past.get(k, 0)) for k in recent}
            top3 = sorted(drifts.items(), key=lambda kv: abs(kv[1]), reverse=True)[:3]
            print(f"[LongTermMemory] top-3 drift over {len(self.history)} sessions: "
                   + ", ".join(f"{k}={v:+.2f}σ" for k, v in top3))
            return pinn_aux       # no-op until training data exist

    ltm = LongTermMemory()
    # Push the current session's anchors (from Section 9) into the history.
    if 'anchors' in dir() and 'VOCAB_COLS' in dir():
        with torch.no_grad():
            sess_anchors_z = anchors.detach().numpy().mean(axis=0)
        ltm.push({c: float(sess_anchors_z[i]) for i, c in enumerate(VOCAB_COLS)})
        print(f"LongTermMemory seeded with this session's anchors: "
               f"{len(ltm.history)} session(s) in memory.")
    print("Step 4 stub installed.  No training data yet — see docstring for requirements.")


### 12b.  Step-4 REPORT — LongTermMemory diagnostic

Snapshot of the long-term-memory state (descriptor history depth + most-drifted descriptors).


In [ ]:
# Step-4 REPORT.
print("=" * 64)
print(f"STEP-4 REPORT: LongTermMemory state")
print("=" * 64)
if 'ltm' not in dir():
    print("[skipped — LongTermMemory instance `ltm` not in scope.]")
else:
    print(f"\n  history depth     : {len(ltm.history)} session(s)")
    print(f"  config.update_threshold: {ltm.cfg.update_threshold}")
    print(f"  config.history_window:   {ltm.cfg.history_window}")
    if len(ltm.history) >= 1:
        last = ltm.history[-1]
        sorted_anchors = sorted(last.items(), key=lambda kv: abs(kv[1]), reverse=True)
        print(f"\n  Most-extreme anchors this session (|z| sort):")
        for k, v in sorted_anchors[:5]:
            print(f"    {k:14s} = {v:+.2f} σ")
    print("\n  data-requirements (to enable training):")
    print("    1. >=30 repeated FD-curve sessions per (probe, sample).")
    print("    2. Wall-clock or contact-count metadata per session.")
    print("    3. Optional probe-replacement events (anchors the reset classifier).")
print("\n" + "=" * 64 + "\n")


## 13.  Transfer-learning workflow

Three helpers for adapting the framework to a new probe / sample combination:

1. `save_pinn_checkpoint()` — bundles the trained PINN state with provenance
   (probe, sample, vocab columns, train/test FD indices, μ/σ stats).
2. `load_pinn_checkpoint(path)` — restores the architecture + weights.
3. `freeze_backbone_finetune_heads(model)` — freezes the CNN backbone and
   leaves only the anchor head / param head / auxiliary head trainable, so
   that fine-tuning on new (probe, sample) data uses ~10× fewer epochs.

A short worked example at the bottom of the cell shows the swap workflow:
1. Load this notebook's `pinn_step2_model.pt` as a warm start.
2. Update `DATA` in Section 1.2 to point at the new dataset.
3. Re-run cells 1.1 → 9; the warm-started PINN will converge in ~150 epochs
   instead of 1500.

This is the recommended pattern for moving the framework across probes /
samples without re-training from scratch.


In [ ]:
# Transfer-learning helpers.
if TORCH_OK:
    import time as _time

    def save_pinn_checkpoint(model, path=None, extra=None):
        """Save PINN state + provenance for reuse on new data."""
        path = path or (FIG_DIR / f'pinn_checkpoint_{USER.cantilever_id}_{USER.sample_id}.pt')
        payload = {
            'state_dict':  model.state_dict(),
            'probe_vocab': PROBE_VOCAB,
            'sample_vocab':SAMPLE_VOCAB,
            'vocab_cols':  VOCAB_COLS,
            'vocab_weights': VOCAB_WEIGHTS,
            'probe':       USER.cantilever_id,
            'sample':      USER.sample_id,
            'n_fd_curves': int(fd_amp.shape[0]),
            'L_resample':  L_RESAMPLE,
            'saved_at':    _time.strftime('%Y-%m-%dT%H:%M:%S'),
        }
        if extra: payload['extra'] = extra
        torch.save(payload, path)
        print(f"PINN checkpoint saved: {path}")
        return path

    def load_pinn_checkpoint(path):
        """Load a saved PINN payload and rebuild the model.

        Returns (model, payload).  The caller is responsible for verifying
        that the payload's vocab + (probe, sample) tokens are compatible
        with the current notebook's DATA.
        """
        payload = torch.load(path, map_location='cpu')
        model = FDEncoderPINN(n_anchors=len(payload['vocab_cols']),
                                n_params=5,
                                n_probe=len(payload['probe_vocab']),
                                n_sample=len(payload['sample_vocab']),
                                L=payload['L_resample'])
        model.load_state_dict(payload['state_dict'])
        print(f"PINN checkpoint loaded from {path}")
        print(f"  trained on: ({payload['probe']}, {payload['sample']}), "
               f"{payload['n_fd_curves']} curves, saved {payload['saved_at']}")
        return model, payload

    def freeze_backbone_finetune_heads(model, *, freeze_aux=False):
        """Freeze the CNN backbone + main MLP; leave only the heads trainable.

        Use after load_pinn_checkpoint() to fine-tune on new (probe, sample) data
        with ~10× fewer epochs than training from scratch.
        """
        for p in model.conv.parameters(): p.requires_grad = False
        for p in model.mlp.parameters():  p.requires_grad = False
        # Heads remain trainable; aux table optionally frozen too.
        if freeze_aux:
            model.aux.requires_grad = False
        n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        n_total     = sum(p.numel() for p in model.parameters())
        print(f"Backbone frozen.  Trainable params: {n_trainable:,} / {n_total:,} "
               f"({100*n_trainable/n_total:.1f}%)")
        return model

    print("Transfer-learning helpers defined: save_pinn_checkpoint, "
           "load_pinn_checkpoint, freeze_backbone_finetune_heads")
    print("\nWORKED EXAMPLE (un-comment to run):")
    print("    # On the source notebook (e.g. Tap300/AlScN), after Section 9 training:")
    print("    #   ckpt_path = save_pinn_checkpoint(model)")
    print("    # On the target notebook (e.g. new probe/sample), instead of training from scratch:")
    print("    #   model, payload = load_pinn_checkpoint(ckpt_path)")
    print("    #   model = freeze_backbone_finetune_heads(model)")
    print("    #   opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=5e-4)")
    print("    #   # then run ~150 epochs of the training loop in cell 39")

    # Auto-save the current run's checkpoint, if model exists.
    _pinn_to_save = pinn_model if 'pinn_model' in dir() else (model if 'model' in dir() else None)
    if isinstance(_pinn_to_save, FDEncoderPINN):
        try:
            save_pinn_checkpoint(_pinn_to_save)
        except Exception as e:
            print(f"  (skipped auto-save: {e})")
